In [1]:
import os
import json
import pickle
import typing as t
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
import time
import random
import traceback
from dotenv import load_dotenv
from openai import AzureOpenAI
import asyncio
from itertools import combinations
import re
import logging


## langchain import
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document as LCDocument
from langchain_core.embeddings import Embeddings
from langchain_openai import AzureChatOpenAI
from langchain_core.callbacks import BaseCallbackManager, Callbacks
from langchain_core.documents import Document as LCDocument
from langchain_core.embeddings import Embeddings as LangchainEmbeddings
from langchain_core.language_models import BaseLanguageModel as LangchainLLM

## ragas import 
from ragas.testset.graph import Relationship
from ragas.testset.graph import Node
from ragas.testset.transforms.base import Extractor
from ragas.testset.transforms.base import RelationshipBuilder
from ragas.testset.graph import NodeType
from ragas.testset.transforms.engine import Parallel
from ragas.testset.transforms.extractors import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors.llm_based import ThemesExtractor
from ragas.testset.transforms.relationship_builders import (
    CosineSimilarityBuilder,
    OverlapScoreBuilder,
)
from ragas._analytics import TestsetGenerationEvent, track
from ragas.callbacks import new_group
from ragas.cost import TokenUsageParser
from ragas.embeddings.base import BaseRagasEmbeddings, LangchainEmbeddingsWrapper
from ragas.executor import Executor
from ragas.llms.base import BaseRagasLLM
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, LLMResult
import asyncio
from ragas.run_config import RunConfig
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.persona import Persona, generate_personas_from_kg
from ragas.testset.synthesizers import QueryDistribution, default_query_distribution
from ragas.testset.synthesizers.base import BaseScenario
from ragas.testset.synthesizers.testset_schema import Testset, TestsetSample
from ragas.testset.synthesizers.utils import calculate_split_values
from ragas.testset.transforms import apply_transforms



repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "chatbot").exists() and (candidate / ".env").exists():
        repo_root = candidate
        break
load_dotenv(repo_root / ".env", override=True)

print("repo_root =", repo_root)

/opt/anaconda3/envs/gds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo_root = /Users/senghok/Documents/gds_tender_test


In [2]:
file_names_excluded = [
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0011_Tender Supplement #5 - Updated Design Dossier (Aug25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0002_Tender Supplement #2 Update Design Dossier #2 & Contract Documents #1(Apr25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0008_Design Dossier Minor Update.pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0003_Tender Supplement #3 - Updated Design Dossier #3  Contract Documents (Exhibits)  Information response templates #2 (End April2025).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0005_Tender Supplement #4 - Updated Design Dossier (EndMay25).pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part B - Input_Exhibit B - Compensation_Rev2_20250808.xlsx',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part B - Input_Exhibit B - Compensation_Rev1_20250430.xlsx',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0009_Company_FEED_Value_Enhancement_Optimisation_Update_.pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0001_Tender Supplement #1 - Updated Design Dossier (Jan25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0010_Attachment 2_VE Opportunities.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A_Scope of Work_Rev.0_20241216.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 02_Schedule 1_Generald Conditions_Rev.0_20241216.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract_Doc Register as of 08 Aug 25.xlsx',
    'ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Exhibit B Tables - Commercial - Ichthys Onshore AGRU Upgrade and CCS Project FEED validation contract.xlsx', 
]

In [3]:
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_APIM"),
    api_key=os.getenv("API_KEY", "DUMMY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    default_headers={
        "Ocp-Apim-Subscription-Key": os.getenv("APIM_SUBSCRIPTION_KEY")
    },
)

llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_APIM"),
    api_key=os.getenv("API_KEY", "DUMMY"),
    default_headers={
        "Ocp-Apim-Subscription-Key": os.getenv("APIM_SUBSCRIPTION_KEY")
    },
    temperature=0,
    max_retries=1,
    request_timeout=120,
)


In [4]:
class TenderAzureEmbeddings(Embeddings):
    def __init__(
        self,
        model: str = "text-embedding-3-large-llm4t",
        dimensions: int = 3072,
        max_retries: int = 6,
        retry_sleep_seconds: float = 2.0,
    ):
        self.model = model
        self.dimensions = dimensions
        self.max_retries = max_retries
        self.retry_sleep_seconds = retry_sleep_seconds

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                response = openai_client.embeddings.create(
                    model=self.model,
                    input=texts,
                    dimensions=self.dimensions
                )
                return [item.embedding for item in response.data]

            except Exception as e:
                last_error = e
                print(
                    f"[embedding-retry] attempt {attempt}/{self.max_retries} failed: "
                    f"{type(e).__name__}: {e}"
                )

                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * (2 ** (attempt - 1))
                    wait += random.uniform(0, 0.5)
                    time.sleep(wait)

        raise RuntimeError(
            f"Embedding failed after {self.max_retries} attempts"
        ) from last_error

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]


embedding_model = TenderAzureEmbeddings()

In [6]:
# # ---------------------------------------------------------------------
# # Dummy embedding test
# # ---------------------------------------------------------------------

# dummy_texts = [
#     "This is a test sentence for embedding.",
#     "The Contractor shall submit the HSES management plan for Company review.",
#     "Exhibit A13 defines contractor documentation deliverables.",
# ]

# embedding_model = TenderAzureEmbeddings()

# print("[test] model =", embedding_model.model)
# print("[test] dimensions =", embedding_model.dimensions)

# # Test single query
# vec = embedding_model.embed_query(dummy_texts[0])

# print("\n[single query]")
# print("type(vec) =", type(vec))
# print("len(vec) =", len(vec))
# print("first 5 values =", vec[:5])
# print("value types =", [type(x).__name__ for x in vec[:5]])

# if isinstance(vec, str):
#     print("ERROR: embedding is a string/base64, not a float vector")
# elif len(vec) != embedding_model.dimensions:
#     print(f"ERROR: expected {embedding_model.dimensions}, got {len(vec)}")
# else:
#     print("OK: single embedding looks valid")

# # Test batch
# vectors = embedding_model.embed_documents(dummy_texts)

# print("\n[batch]")
# print("number of vectors =", len(vectors))
# print("dimensions =", [len(v) for v in vectors])
# print("first vector first 5 values =", vectors[0][:5])

# all_valid = (
#     len(vectors) == len(dummy_texts)
#     and all(isinstance(v, list) for v in vectors)
#     and all(len(v) == embedding_model.dimensions for v in vectors)
#     and all(isinstance(x, float) for v in vectors for x in v[:10])
# )

# print("batch valid =", all_valid)

In [7]:

@dataclass
class DirectSummaryEmbeddingExtractor(Extractor):
    embedding_model: t.Any = None
    property_name: str = "summary_embedding"
    embed_property_name: str = "summary"

    async def extract(self, node: Node):
        text = (node.get_property(self.embed_property_name) or "").strip()

        if not text:
            return self.property_name, None

        metadata = node.get_property("document_metadata") or {}

        try:
            vector = self.embedding_model.embed_query(text)
            return self.property_name, vector

        except Exception as e:
            print("\n[embedding-error] failed node")
            print("chunk_id:", metadata.get("chunk_id"))
            print("file_name:", metadata.get("file_name"))
            print("page_number:", metadata.get("page_number"))
            print("summary preview:", text[:1000])
            print("error:", repr(e))
            traceback.print_exc()
            raise

In [5]:
# # ---------------------------------------------------------------------
# # Dummy test for DirectSummaryEmbeddingExtractor
# # ---------------------------------------------------------------------

# from ragas.testset.graph import Node, NodeType

# dummy_node = Node(
#     type=NodeType.CHUNK,
#     properties={
#         "summary": "The Contractor shall submit the HSES management plan for Company review.",
#         "document_metadata": {
#             "chunk_id": "dummy_chunk_001",
#             "file_name": "dummy_tender.pdf",
#             "page_number": 1,
#         },
#     },
# )

# empty_summary_node = Node(
#     type=NodeType.CHUNK,
#     properties={
#         "summary": "",
#         "document_metadata": {
#             "chunk_id": "dummy_chunk_empty",
#             "file_name": "dummy_tender.pdf",
#             "page_number": 2,
#         },
#     },
# )

# extractor = DirectSummaryEmbeddingExtractor(
#     embedding_model=embedding_model,
#     property_name="summary_embedding",
#     embed_property_name="summary",
# )

# # Test normal node
# prop_name, vector = await extractor.extract(dummy_node)

# print("[normal node]")
# print("prop_name =", prop_name)
# print("type(vector) =", type(vector))
# print("len(vector) =", len(vector) if vector is not None else None)
# print("first 5 values =", vector[:5] if vector else None)
# print("value types =", [type(x).__name__ for x in vector[:5]] if vector else None)

# if prop_name != "summary_embedding":
#     print("ERROR: wrong property name")
# elif vector is None:
#     print("ERROR: vector is None")
# elif isinstance(vector, str):
#     print("ERROR: vector is string/base64, not list[float]")
# elif len(vector) != embedding_model.dimensions:
#     print(f"ERROR: expected {embedding_model.dimensions}, got {len(vector)}")
# elif not all(isinstance(x, float) for x in vector[:10]):
#     print("ERROR: vector values are not floats")
# else:
#     print("OK: extractor produced valid embedding")

# # Test empty summary node
# empty_prop_name, empty_vector = await extractor.extract(empty_summary_node)

# print("\n[empty summary node]")
# print("prop_name =", empty_prop_name)
# print("vector =", empty_vector)

# if empty_prop_name == "summary_embedding" and empty_vector is None:
#     print("OK: empty summary returns None")
# else:
#     print("ERROR: unexpected empty-summary behavior")

### Load state and build Ragas Chunks with existing KG Entities

In [8]:
STATE_FILE = repo_root / "output/jupyter-notebook/checkpoints/state_closed_v5_merge.pkl"

with open(STATE_FILE, "rb") as f:
    state = pickle.load(f)

entity_by_key = {
    ent["entity_key"]: ent
    for ent in state["all_entity_nodes"]
    if ent.get("entity_key")
}

chunk_entities_by_id = defaultdict(list)

for mention in state["all_mentions"]:
    chunk_id = mention.get("chunk_id")
    entity_key = mention.get("entity_key")

    ent = entity_by_key.get(entity_key)
    if not chunk_id or not ent:
        continue

    chunk_entities_by_id[chunk_id].append({
        "name": ent.get("name"),
        "canonical_category": ent.get("canonical_category"),
        "llm_category": ent.get("llm_category"),
        "description": ent.get("description"),
    })

ragas_chunks = []

for ch in state["all_chunk_payloads"]:
    chunk_id = ch.get("chunk_id")
    text = (ch.get("text") or "").strip()

    if not text:
        continue

    ragas_chunks.append(
        LCDocument(
            page_content=text,
            metadata={
                "chunk_id": chunk_id,
                "file_name": ch.get("file_name"),
                "page_number": ch.get("page_number"),
                "doc_url": ch.get("doc_url"),
                "sequence_number": ch.get("sequence_number"),
                "section_hint": ch.get("section_hint"),
                "entities": chunk_entities_by_id.get(chunk_id, []),
            },
        )
    )

print("Ragas chunks:", len(ragas_chunks))
print("Sample metadata:", ragas_chunks[0].metadata)
print("Sample text:", ragas_chunks[0].page_content[:300])

Ragas chunks: 1555
Sample metadata: {'chunk_id': 'ROC_INPEX_2_0_Tender_Supplements_WS33071728_-TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update_pdf_size_5858729_page_1_text_chunk_1', 'file_name': 'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update.pdf', 'page_number': '1', 'doc_url': 'file://c:\\Users\\hseng\\Documents\\gds_tender_test\\ROC_INPEX\\2.0 Tender Supplements\\WS33071728 -TB-0004_Company_FEED_Value_Enhancement_Optimisation_Update.pdf', 'sequence_number': 1, 'section_hint': None, 'entities': [{'name': 'Value Enhancement (VE) process', 'canonical_category': 'other', 'llm_category': 'value_enhancement_process', 'description': 'An internal process initiated by the Company to identify, assess, and select potential value enhancement opportunities for the project.'}, {'name': 'Category 1', 'canonical_category': 'requirement', 'llm_category': 'enhancement_category', 'description': 'Potential opportunities to be developed

In [9]:


def is_azure_content_filter_error(e: Exception) -> bool:
    msg = str(e).lower()
    return (
        "content filter" in msg
        or "content_filter" in msg
        or "responsibleaipolicyviolation" in msg
        or "azure has not provided the response" in msg
    )


@dataclass
class DirectTenderSummaryExtractor(Extractor):
    llm: t.Any = None
    property_name: str = "summary"
    max_chars: int = 5000
    max_retries: int = 3
    retry_sleep_seconds: float = 2.0

    def _messages(self, text: str):
        user_prompt = f"""
Summarize the given tender text in less than 10 sentences.

Do not write a generic summary.
Prefer a compact factual summary that can support synthetic question generation.
If the text is mainly a list or table, summarize the concrete listed items and their purpose.

Text:
{text[:self.max_chars]}
""".strip()

        return [
            SystemMessage(content=(
                "You are a senior tendering and procurement knowledge-graph curator. "
                "Your summaries must be factual, compact, domain-specific, and useful for "
                "synthetic question generation. Do not invent facts."
            )),
            HumanMessage(content=user_prompt),
        ]

    async def _summarize_once(self, text: str) -> str:
        result = await self.llm.ainvoke(self._messages(text))
        return (result.content or "").strip()

    async def extract(self, node: Node):
        text = (node.get_property("page_content") or "").strip()

        if not text:
            return self.property_name, ""

        metadata = node.get_property("document_metadata") or {}
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                summary = await self._summarize_once(text)
                return self.property_name, summary

            except Exception as e:
                last_error = e

                if not is_azure_content_filter_error(e):
                    raise

                print(
                    f"[content-filter retry] DirectTenderSummaryExtractor "
                    f"attempt {attempt}/{self.max_retries}"
                )
                print("chunk_id:", metadata.get("chunk_id"))
                print("file_name:", metadata.get("file_name"))
                print("page_number:", metadata.get("page_number"))

                if attempt < self.max_retries:
                    await asyncio.sleep(self.retry_sleep_seconds)

        raise last_error

In [10]:

@dataclass
class DirectTenderThemesExtractor(Extractor):
    llm: t.Any = None
    property_name: str = "themes"
    max_themes: int = 10
    max_chars: int = 5000

    async def extract(self, node: Node):
        text = (node.get_property("page_content") or "").strip()

        if not text:
            return self.property_name, []

        user_prompt = f"""
Extract the main themes and concepts from the given text.

Return only valid JSON in this format:
{{
  "themes": ["theme 1", "theme 2"]
}}

Limit:
- maximum {self.max_themes} themes
- short phrases only

Text:
{text[:self.max_chars]}
""".strip()

        result = await self.llm.ainvoke([
            SystemMessage(content= "You are a senior EPC tendering and procurement knowledge-graph curator. "
                                    "You summarize tendering, ITB, EPC contract, procurement, technical, commercial, "
                                    "HSES, quality, construction, and project controls documents. "),
            HumanMessage(content=user_prompt),
        ])
        raw = result.content.strip()
        try:
            data = json.loads(raw)
            themes = data.get("themes", [])
        except Exception:
            themes = [
                line.strip("-• ").strip()
                for line in raw.splitlines()
                if line.strip()
            ]
        clean = []
        seen = set()
        for theme in themes:
            theme = str(theme).strip()
            if not theme:
                continue
            key = theme.lower()
            if key in seen:
                continue
            clean.append(theme)
            seen.add(key)
            if len(clean) >= self.max_themes:
                break
        return self.property_name, clean

In [11]:
import math
def cosine_similarity(a, b):
    if not a or not b:
        return 0.0

    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))

    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0

    return dot / (norm_a * norm_b)

In [12]:

@dataclass
class DirectCosineSimilarityBuilder(RelationshipBuilder):
    property_name: str = "summary_embedding"
    new_property_name: str = "summary_similarity"
    threshold: float = 0.7

    async def transform(self, kg):
        relationships = []
        nodes = self.filter(kg).nodes

        for node_x, node_y in combinations(nodes, 2):
            emb_x = node_x.get_property(self.property_name)
            emb_y = node_y.get_property(self.property_name)

            if emb_x is None or emb_y is None:
                continue

            score = cosine_similarity(emb_x, emb_y)

            if score >= self.threshold:
                relationships.append(
                    Relationship(
                        source=node_x,
                        target=node_y,
                        type=self.new_property_name,
                        properties={self.new_property_name: float(score)},
                        bidirectional=True,
                    )
                )

        return relationships

    def generate_execution_plan(self, kg):
        async def find_and_add_relationships():
            relationships = await self.transform(kg)
            kg.relationships.extend(relationships)

        return [find_and_add_relationships()]

In [26]:
def normalized_token_set(text):
    return {
        token
        for token in re.sub(r"[^a-z0-9]+", " ", text.lower()).split()
        if len(token) >= 3
    }


from rapidfuzz.distance import JaroWinkler

def entity_similarity(a: str, b: str) -> float:
    return JaroWinkler.similarity(a.lower(), b.lower())

In [27]:
@dataclass
class DirectEntityOverlapBuilder(RelationshipBuilder):
    property_name: str = "entities"
    new_property_name: str = "overlap_score"
    threshold: float = 0.05
    entity_similarity_threshold: float = 0.75
    noisy_top_percent: float = 0.05
    ## get the most common
    def get_noisy_items(self, nodes):
        from collections import Counter
        all_items = []
        for node in nodes:
            items = node.get_property(self.property_name) or []
            all_items.extend([str(x).strip() for x in items if str(x).strip()])

        if not all_items:
            return set()
        counts = Counter(all_items)
        n = max(1, int(len(counts) * self.noisy_top_percent))
        return {item for item, _ in counts.most_common(n)}

    async def transform(self, kg):
        relationships = []
        nodes = self.filter(kg).nodes
        noisy_items = self.get_noisy_items(nodes)

        for node_x, node_y in combinations(nodes, 2):
            x_items = node_x.get_property(self.property_name) or []
            y_items = node_y.get_property(self.property_name) or []
            x_items = [x for x in x_items if x not in noisy_items]
            y_items = [y for y in y_items if y not in noisy_items]

            if not x_items or not y_items:
                continue
            comparisons = 0
            matches = 0
            overlapped_items = []
            for x in x_items:
                for y in y_items:
                    comparisons += 1
                    score = entity_similarity(x, y)
                    if score >= self.entity_similarity_threshold:
                        matches += 1
                        overlapped_items.append((x, y))

            overlap_score = matches / comparisons if comparisons else 0.0

            if overlap_score >= self.threshold:
                ## create relationship between the two nodes
                relationships.append(
                    Relationship(
                        source=node_x,
                        target=node_y,
                        type=f"{self.property_name}_overlap",
                        properties={
                            f"{self.property_name}_{self.new_property_name}": float(overlap_score),
                            "overlapped_items": overlapped_items,
                        },
                        bidirectional=True,
                    )
                )
        return relationships
    def generate_execution_plan(self, kg):
        async def find_and_add_relationships():
            relationships = await self.transform(kg)
            kg.relationships.extend(relationships)
        return [find_and_add_relationships()]

#### Customized Ragas NER 

In [28]:

@dataclass
class ExistingChunkEntitiesExtractor(Extractor):
    property_name: str = "entities"
    max_entities: int = 30
    async def extract(self, node: Node):
        metadata = node.get_property("document_metadata") or {}
        raw_entities = metadata.get("entities") or []
        names = []
        seen = set()

        for ent in raw_entities:
            if isinstance(ent, dict):
                name = (ent.get("name") or "").strip()
            else:
                name = str(ent).strip()
            if not name:
                continue
            key = name.lower()
            if key in seen:
                continue
            names.append(name)
            seen.add(key)
            if len(names) >= self.max_entities:
                break
        return self.property_name, names

### Customized Tender Transform Pipeline

In [29]:
def tender_transforms_for_prechunked_direct(llm, embedding_model):
    def filter_chunks(node):
        return node.type == NodeType.CHUNK

    summary_extractor = DirectTenderSummaryExtractor(
    llm=llm,
    max_chars=5000,
    max_retries=5,
    retry_sleep_seconds=3.0,
    filter_nodes=filter_chunks,
)

    summary_emb_extractor = DirectSummaryEmbeddingExtractor(
        embedding_model=embedding_model,
        property_name="summary_embedding",
        embed_property_name="summary",
        filter_nodes=filter_chunks,
    )

    theme_extractor = DirectTenderThemesExtractor(
        llm=llm,
        filter_nodes=filter_chunks,
    )

    entity_extractor = ExistingChunkEntitiesExtractor(
        filter_nodes=filter_chunks,
        max_entities=30,
    )

    cosine_sim_builder = DirectCosineSimilarityBuilder(
        property_name="summary_embedding",
        new_property_name="summary_similarity",
        threshold=0.7,
        filter_nodes=filter_chunks,
    )

    entity_overlap_builder = DirectEntityOverlapBuilder(
        property_name="entities",
        threshold=0.05,
        entity_similarity_threshold=0.75,
        filter_nodes=filter_chunks,
    )

    return [
        summary_extractor,
        Parallel(summary_emb_extractor, theme_extractor, entity_extractor),
        Parallel(cosine_sim_builder, entity_overlap_builder),
    ]

In [30]:

DEBUG_RAGAS_TESTSET_GENERATION_GROUP_NAME = "debug ragas testset generation"


class FastRagasLangchainLLM(BaseRagasLLM):
    """
    Ragas-compatible LLM adapter that preserves Ragas prompts/synthesizers but
    calls the underlying LangChain chat model through invoke/ainvoke directly.

    This avoids the slower generate_prompt/agenerate_prompt path while still
    returning the LLMResult shape expected by Ragas PydanticPrompt and
    synthesizer code.
    """

    def __init__(
        self,
        langchain_llm: LangchainLLM,
        run_config: t.Optional[RunConfig] = None,
        cache: t.Any = None,
    ):
        super().__init__(cache=cache)
        self.langchain_llm = langchain_llm
        self.set_run_config(run_config or RunConfig())

    def _prompt_to_input(self, prompt):
        if hasattr(prompt, "to_messages"):
            return prompt.to_messages()
        if hasattr(prompt, "to_string"):
            return prompt.to_string()
        return str(prompt)

    def _bound_llm(self, temperature=None, stop=None):
        bind_kwargs = {}
        if temperature is not None:
            bind_kwargs["temperature"] = temperature
        if stop:
            bind_kwargs["stop"] = stop
        if bind_kwargs and hasattr(self.langchain_llm, "bind"):
            try:
                return self.langchain_llm.bind(**bind_kwargs)
            except Exception:
                return self.langchain_llm
        return self.langchain_llm

    def _invoke_kwargs(self, callbacks=None):
        kwargs = {}
        if callbacks:
            kwargs["config"] = {"callbacks": callbacks}
        return kwargs

    @staticmethod
    def _response_text(response) -> str:
        if hasattr(response, "content"):
            content = response.content
        else:
            content = response
        if isinstance(content, list):
            return "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            ).strip()
        return str(content or "").strip()

    @staticmethod
    def _to_llm_result(texts: t.List[str]) -> LLMResult:
        return LLMResult(
            generations=[
                [ChatGeneration(message=AIMessage(content=text)) for text in texts]
            ]
        )

    def generate_text(
        self,
        prompt,
        n: int = 1,
        temperature: t.Optional[float] = 0.01,
        stop: t.Optional[t.List[str]] = None,
        callbacks: Callbacks = None,
    ) -> LLMResult:
        prompt_input = self._prompt_to_input(prompt)
        call_llm = self._bound_llm(temperature=temperature, stop=stop)
        kwargs = self._invoke_kwargs(callbacks=callbacks)
        texts = []
        for _ in range(max(1, n)):
            response = call_llm.invoke(prompt_input, **kwargs)
            texts.append(self._response_text(response))
        return self._to_llm_result(texts)

    async def agenerate_text(
        self,
        prompt,
        n: int = 1,
        temperature: t.Optional[float] = 0.01,
        stop: t.Optional[t.List[str]] = None,
        callbacks: Callbacks = None,
    ) -> LLMResult:
        prompt_input = self._prompt_to_input(prompt)
        call_llm = self._bound_llm(temperature=temperature, stop=stop)
        kwargs = self._invoke_kwargs(callbacks=callbacks)

        async def _one():
            response = await call_llm.ainvoke(prompt_input, **kwargs)
            return self._response_text(response)

        texts = await asyncio.gather(*[_one() for _ in range(max(1, n))])
        return self._to_llm_result(list(texts))

    def is_finished(self, response: LLMResult) -> bool:
        # Direct invoke/ainvoke returns one complete message per request. If a
        # provider truncates, Ragas retry/validation will usually catch it as a
        # parse failure at the prompt layer.
        return True

    def __repr__(self) -> str:
        return f"FastRagasLangchainLLM(langchain_llm={self.langchain_llm.__class__.__name__}(...))"


@dataclass
class DebugTestsetGenerator:
    """
    Notebook-local copy of Ragas TestsetGenerator for debugging.
    This copies the generate_with_chunks() and generate() control flow into this notebook,
    so you can add prints, breakpoints, or inspect intermediate KnowledgeGraph/scenario state
    without jumping across the Ragas package.
    """
    llm: BaseRagasLLM
    embedding_model: BaseRagasEmbeddings
    knowledge_graph: KnowledgeGraph = field(default_factory=KnowledgeGraph)
    persona_list: t.Optional[t.List[Persona]] = None
    llm_context: t.Optional[str] = None

    @classmethod
    def from_langchain(
        cls,
        llm: LangchainLLM,
        embedding_model: LangchainEmbeddings,
        knowledge_graph: t.Optional[KnowledgeGraph] = None,
        llm_context: t.Optional[str] = None,
    ) -> "DebugTestsetGenerator":
        knowledge_graph = knowledge_graph or KnowledgeGraph()
        return cls(
            FastRagasLangchainLLM(llm),
            LangchainEmbeddingsWrapper(embedding_model),
            knowledge_graph,
            llm_context=llm_context,
        )

  
    def generate_with_chunks(
        self,
        chunks: t.Sequence[t.Union[LCDocument, str]],
        testset_size: int,
        transforms: t.Optional[Transforms] = None,
        transforms_llm: t.Optional[BaseRagasLLM] = None,
        transforms_embedding_model: t.Optional[BaseRagasEmbeddings] = None,
        query_distribution: t.Optional[QueryDistribution] = None,
        run_config: t.Optional[RunConfig] = None,
        callbacks: t.Optional[Callbacks] = None,
        token_usage_parser: t.Optional[TokenUsageParser] = None,
        with_debugging_logs: bool = False,
        raise_exceptions: bool = True,
        return_executor: bool = False,
        debug: bool = True,
    ) -> t.Union[Testset, Executor]:
        """
        Local copy of Ragas generate_with_chunks().

        Flow:
        chunks -> NodeType.CHUNK nodes -> KnowledgeGraph -> transforms -> self.generate().
        """
        if debug:
            print("[debug] generate_with_chunks: input chunks =", len(chunks))
            print("[debug] generate_with_chunks: testset_size =", testset_size)

        # Force caller to provide LLM + embeddings unless fully custom transforms are supplied.
        if not self.llm and not transforms_llm:
            raise ValueError(
                "An llm client was not provided. Provide an LLM on DebugTestsetGenerator "
                "or as transforms_llm, or provide your own transforms."
            )
        if not self.embedding_model and not transforms_embedding_model:
            raise ValueError(
                "An embedding client was not provided. Provide an embedding model on "
                "DebugTestsetGenerator or as transforms_embedding_model, or provide custom transforms."
            )

        if transforms is None:
            if debug:
                print("[debug] generate_with_chunks: building default_transforms_for_prechunked")
            transforms = default_transforms_for_prechunked(
                llm=transforms_llm or self.llm,
                embedding_model=transforms_embedding_model or self.embedding_model,
            )
        
        # Convert pre-chunked input to Ragas CHUNK nodes.
        nodes = []
        skipped_empty = 0
        for idx, chunk in enumerate(chunks):
            if isinstance(chunk, str):
                page_content = chunk
                metadata = {}
            else:
                page_content = chunk.page_content
                metadata = chunk.metadata

            if page_content is not None and page_content.strip() != "":
                node = Node(
                    type=NodeType.CHUNK,
                    properties={
                        "page_content": page_content,
                        "document_metadata": metadata,
                    },
                )
                nodes.append(node)
            else:
                skipped_empty += 1

        kg = KnowledgeGraph(nodes=nodes)

        if debug:
            print("[debug] generate_with_chunks: KG initial nodes =", len(kg.nodes))
            print("[debug] generate_with_chunks: skipped empty chunks =", skipped_empty)
            print("[debug] generate_with_chunks: applying transforms...")

        apply_transforms(kg, transforms, run_config=run_config or RunConfig())
        self.knowledge_graph = kg

        if debug:
            print("[debug] generate_with_chunks: KG after transforms")
            print("        nodes =", len(self.knowledge_graph.nodes))
            print("        relationships =", len(self.knowledge_graph.relationships))
            prop_counts = {}
            for node in self.knowledge_graph.nodes:
                for key in node.properties:
                    prop_counts[key] = prop_counts.get(key, 0) + 1
            print("        node property coverage =", dict(sorted(prop_counts.items())))

        return self.generate(
            testset_size=testset_size,
            query_distribution=query_distribution,
            run_config=run_config,
            callbacks=callbacks,
            token_usage_parser=token_usage_parser,
            with_debugging_logs=with_debugging_logs,
            raise_exceptions=raise_exceptions,
            return_executor=return_executor,
            debug=debug,
        )

    def generate(
        self,
        testset_size: int,
        query_distribution: t.Optional[QueryDistribution] = None,
        num_personas: int = 3,
        run_config: t.Optional[RunConfig] = None,
        batch_size: t.Optional[int] = None,
        callbacks: t.Optional[Callbacks] = None,
        token_usage_parser: t.Optional[TokenUsageParser] = None,
        with_debugging_logs: bool = False,
        raise_exceptions: bool = True,
        return_executor: bool = False,
        debug: bool = True,
    ) -> t.Union[Testset, Executor]:
        """
        Local copy of Ragas generate().

        Flow:
        existing KnowledgeGraph -> personas -> scenarios -> samples -> Testset.
        """

        if run_config is not None and isinstance(self.llm, BaseRagasLLM):
            self.llm.set_run_config(run_config)

        query_distribution = query_distribution or default_query_distribution(
            self.llm, self.knowledge_graph, self.llm_context
        )
        callbacks = callbacks or []

        if debug:
            print("[debug] generate: testset_size =", testset_size)
            print("[debug] generate: query_distribution =")
            for synthesizer, probability in query_distribution:
                print("        ", getattr(synthesizer, "name", synthesizer.__class__.__name__), probability)

        ragas_callbacks = {}
        if token_usage_parser is not None:
            from ragas.cost import CostCallbackHandler

            cost_cb = CostCallbackHandler(token_usage_parser=token_usage_parser)
            ragas_callbacks["cost_cb"] = cost_cb
        else:
            cost_cb = None

        for cb in ragas_callbacks.values():
            if isinstance(callbacks, BaseCallbackManager):
                callbacks.add_handler(cb)
            else:
                callbacks.append(cb)

        testset_generation_rm, testset_generation_grp = new_group(
            name=DEBUG_RAGAS_TESTSET_GENERATION_GROUP_NAME,
            inputs={"testset_size": testset_size},
            callbacks=callbacks,
        )

        if with_debugging_logs:
            from ragas.utils import patch_logger

            patch_logger("ragas.experimental.testset.synthesizers", logging.DEBUG)
            patch_logger("ragas.experimental.testset.graph", logging.DEBUG)
            patch_logger("ragas.experimental.testset.transforms", logging.DEBUG)

        if self.persona_list is None:
            if debug:
                print("[debug] generate: generating personas from KG")
            self.persona_list = generate_personas_from_kg(
                llm=self.llm,
                kg=self.knowledge_graph,
                num_personas=num_personas,
                callbacks=callbacks,
            )
        else:
            if debug:
                print("[debug] generate: using existing persona_list")
            random.shuffle(self.persona_list)

        if debug:
            print("[debug] generate: personas =")
            for persona in self.persona_list[:num_personas]:
                print("        ", persona)

        splits, _ = calculate_split_values(
            [prob for _, prob in query_distribution], testset_size
        )

        if debug:
            print("[debug] generate: scenario splits =", splits)

        scenario_generation_rm, scenario_generation_grp = new_group(
            name="Scenario Generation",
            inputs={"splits": splits},
            callbacks=testset_generation_grp,
        )

        exec = Executor(
            desc="Generating Scenarios",
            raise_exceptions=raise_exceptions,
            run_config=run_config,
            keep_progress_bar=False,
            batch_size=batch_size,
        )

        splits, _ = calculate_split_values(
            [prob for _, prob in query_distribution], testset_size
        )
        for i, (scenario, _) in enumerate(query_distribution):
            if debug:
                print(
                    "[debug] generate: submit scenario generation",
                    getattr(scenario, "name", scenario.__class__.__name__),
                    "n=",
                    splits[i],
                )
            exec.submit(
                scenario.generate_scenarios,
                n=splits[i],
                knowledge_graph=self.knowledge_graph,
                persona_list=self.persona_list[:num_personas],
                callbacks=scenario_generation_grp,
            )

        try:
            scenario_sample_list: t.List[t.List[BaseScenario]] = exec.results()
        except Exception as e:
            scenario_generation_rm.on_chain_error(e)
            raise e
        else:
            scenario_generation_rm.on_chain_end(
                outputs={"scenario_sample_list": scenario_sample_list}
            )

        if debug:
            print("[debug] generate: scenario_sample_list sizes =", [len(x) for x in scenario_sample_list])

        sample_generation_rm, sample_generation_grp = new_group(
            name="Sample Generation",
            inputs={"scenario_sample_list": scenario_sample_list},
            callbacks=testset_generation_grp,
        )
        exec = Executor(
            "Generating Samples",
            raise_exceptions=raise_exceptions,
            run_config=run_config,
            keep_progress_bar=True,
            batch_size=batch_size,
        )

        additional_testset_info: t.List[t.Dict] = []
        for i, (synthesizer, _) in enumerate(query_distribution):
            for sample in scenario_sample_list[i]:
                exec.submit(
                    synthesizer.generate_sample,
                    scenario=sample,
                    callbacks=sample_generation_grp,
                )
                additional_testset_info.append(
                    {"synthesizer_name": synthesizer.name}
                )

        if return_executor:
            if debug:
                print("[debug] generate: returning Executor before sample generation results")
            return exec

        try:
            eval_samples = exec.results()
        except Exception as e:
            sample_generation_rm.on_chain_error(e)
            raise e
        else:
            sample_generation_rm.on_chain_end(outputs={"eval_samples": eval_samples})

        testsets = []
        for sample, additional_info in zip(eval_samples, additional_testset_info):
            testsets.append(TestsetSample(eval_sample=sample, **additional_info))

        testset = Testset(samples=testsets, cost_cb=cost_cb)
        testset_generation_rm.on_chain_end({"testset": testset})
        track(
            TestsetGenerationEvent(
                event_type="testset_generation",
                evolution_names=[
                    e.__class__.__name__.lower() for e, _ in query_distribution
                ],
                evolution_percentages=[p for _, p in query_distribution],
                num_rows=testset_size,
                language="english",
            )
        )
        if debug:
            print("[debug] generate: final samples =", len(testset.samples))

        return testset


In [33]:
generator = DebugTestsetGenerator.from_langchain(
    llm=llm,
    embedding_model=embedding_model,
    llm_context=(
        "Tendering, ITB, EPC contract, procurement, technical requirements, "
        "commercial requirements, HSES, quality, construction, project controls."
    ),
)
transforms = tender_transforms_for_prechunked_direct(
    llm=llm,
    embedding_model=embedding_model,
)

/var/folders/_9/m2ryk39s2slc67mc6y5xdxcm0000gn/T/ipykernel_18724/2969013362.py:142: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  LangchainEmbeddingsWrapper(embedding_model),


In [34]:
# ---------------------------------------------------------------------
# RAGAS KG CACHE: build once, reuse for data generation
# ---------------------------------------------------------------------


RAGAS_KG_CACHE_DIR = repo_root / "chatbot" / "ragas_kg_cache"
RAGAS_KG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Change the name if you change chunk count or transform thresholds.
RAGAS_KG_CACHE_PATH = RAGAS_KG_CACHE_DIR / "roc_inpex_ragas_kg_full.pkl"


def build_ragas_kg_from_chunks(chunks, transforms, run_config=None, debug=True):
    nodes = []
    skipped_empty = 0

    for chunk in chunks:
        if isinstance(chunk, str):
            page_content = chunk
            metadata = {}
        else:
            page_content = chunk.page_content
            metadata = chunk.metadata

        if page_content and page_content.strip():
            nodes.append(
                Node(
                    type=NodeType.CHUNK,
                    properties={
                        "page_content": page_content,
                        "document_metadata": metadata,
                    },
                )
            )
        else:
            skipped_empty += 1
    kg = KnowledgeGraph(nodes=nodes)
    if debug:
        print("[kg-cache] initial nodes =", len(kg.nodes))
        print("[kg-cache] skipped empty chunks =", skipped_empty)
        print("[kg-cache] applying transforms...")
    apply_transforms(kg, transforms, run_config=run_config or RunConfig())

    if debug:
        prop_counts = {}
        for node in kg.nodes:
            for key in node.properties:
                prop_counts[key] = prop_counts.get(key, 0) + 1
        from collections import Counter
        print("[kg-cache] KG after transforms")
        print("           nodes =", len(kg.nodes))
        print("           relationships =", len(kg.relationships))
        print("           relationship distribution =", dict(Counter(rel.type for rel in kg.relationships)))
        print("           node property coverage =", dict(sorted(prop_counts.items())))

    return kg

def load_or_build_ragas_kg(
    generator,
    chunks,
    transforms,
    cache_path=RAGAS_KG_CACHE_PATH,
    force_rebuild=False,
    debug=True,
):
    cache_path = Path(cache_path)

    if cache_path.exists() and not force_rebuild:
        with cache_path.open("rb") as f:
            kg = pickle.load(f)
        generator.knowledge_graph = kg
        if debug:
            print("[kg-cache] loaded:", cache_path)
            print("[kg-cache] nodes =", len(kg.nodes))
            print("[kg-cache] relationships =", len(kg.relationships))
        return kg

    kg = build_ragas_kg_from_chunks(
        chunks=chunks,
        transforms=transforms,
        debug=debug,
    )

    with cache_path.open("wb") as f:
        pickle.dump(kg, f)

    generator.knowledge_graph = kg
    if debug:
        print("[kg-cache] saved:", cache_path)

    return kg


kg = load_or_build_ragas_kg(
    generator=generator,
    chunks=ragas_chunks,
    transforms=transforms,
    cache_path=RAGAS_KG_CACHE_PATH,
    force_rebuild=False,
    debug=True,
)


[kg-cache] initial nodes = 1555
[kg-cache] skipped empty chunks = 0
[kg-cache] applying transforms...


Applying DirectTenderSummaryExtractor:  56%|█████▌    | 871/1555 [04:18<03:24,  3.35it/s]

[content-filter retry] DirectTenderSummaryExtractor attempt 1/5
chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_03_Exhibit_A8_Completions_Pre-Commissioning_Commissioning_and_Handover_Rev_2_20250808_Clean_pdf_size_585049_page_11_text_chunk_2
file_name: ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A8_Completions Pre-Commissioning Commissioning and Handover_Rev.2_20250808_Clean.pdf
page_number: 11


Applying DirectTenderSummaryExtractor:  60%|█████▉    | 930/1555 [04:34<03:01,  3.44it/s]

[content-filter retry] DirectTenderSummaryExtractor attempt 2/5
chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_03_Exhibit_A8_Completions_Pre-Commissioning_Commissioning_and_Handover_Rev_2_20250808_Clean_pdf_size_585049_page_11_text_chunk_2
file_name: ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A8_Completions Pre-Commissioning Commissioning and Handover_Rev.2_20250808_Clean.pdf
page_number: 11


Applying DirectTenderSummaryExtractor:  62%|██████▏   | 961/1555 [04:45<03:29,  2.84it/s]

[content-filter retry] DirectTenderSummaryExtractor attempt 3/5
chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_03_Exhibit_A8_Completions_Pre-Commissioning_Commissioning_and_Handover_Rev_2_20250808_Clean_pdf_size_585049_page_11_text_chunk_2
file_name: ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A8_Completions Pre-Commissioning Commissioning and Handover_Rev.2_20250808_Clean.pdf
page_number: 11


Applying DirectEntityOverlapBuilder: 100%|██████████| 1/1 [00:13<00:00, 13.41s/it]


[kg-cache] KG after transforms
           nodes = 1555
           relationships = 74607
           relationship distribution = {'summary_similarity': 56683, 'entities_overlap': 17924}
           node property coverage = {'document_metadata': 1555, 'entities': 1555, 'page_content': 1555, 'summary': 1555, 'summary_embedding': 1555, 'themes': 1555}
[kg-cache] saved: /Users/senghok/Documents/gds_tender_test/chatbot/ragas_kg_cache/roc_inpex_ragas_kg_full.pkl


In [ ]:
import random
import math
import typing as t
from collections import Counter
from dataclasses import dataclass

from ragas.testset.graph import KnowledgeGraph, Node
from ragas.testset.persona import Persona, PersonaList
from ragas.testset.synthesizers.base import QueryStyle, QueryLength
from ragas.testset.synthesizers.prompts import ThemesPersonasInput
from ragas.testset.synthesizers.single_hop.base import SingleHopScenario
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.single_hop.base import SingleHopScenario
from ragas.testset.synthesizers.single_hop.prompts import QueryCondition
from ragas.testset.synthesizers.testset_schema import SingleTurnSample


def make_single_hop_context(node) -> str:
    metadata = node.properties.get("document_metadata", {}) or {}

    chunk_id = metadata.get("chunk_id", "")
    file_name = metadata.get("file_name", "")
    page_number = metadata.get("page_number", "")

    page_content = node.properties.get("page_content", "")

    return f"""chunk_id: {chunk_id}
file_name: {file_name}
page_number: {page_number}

{page_content}""".strip()

@dataclass
class RagasLikeSingleHopQuerySynthesizer(SingleHopSpecificQuerySynthesizer):
    name: str = "ragas_like_single_hop_query_synthesizer"
    property_name: str = "entities"
    min_context_chars: int = 1
    max_entities_per_node: int = 30
    debug: bool = True

    def _clean_terms(self, raw_terms) -> list[str]:
        terms = self._extract_themes_from_items(raw_terms)

        clean = []
        seen = set()

        for term in terms:
            term = str(term).strip()
            if not term:
                continue

            key = term.lower()
            if key in seen:
                continue

            clean.append(term)
            seen.add(key)

            if len(clean) >= self.max_entities_per_node:
                break

        return clean

    def get_node_clusters(self, knowledge_graph: KnowledgeGraph) -> list[Node]:
        candidate_nodes = []
        skipped = Counter()

        for node in knowledge_graph.nodes:
            if node.type.name != "CHUNK":
                skipped["not_chunk"] += 1
                continue

            page_content = (node.properties.get("page_content") or "").strip()
            if len(page_content) < self.min_context_chars:
                skipped["empty_or_short_context"] += 1
                continue

            terms = self._clean_terms(node.properties.get(self.property_name, []))
            if not terms:
                skipped["missing_entities"] += 1
                continue

            candidate_nodes.append(node)

        if self.debug:
            print("\n[ragas-like-single-hop] candidate node stats")
            print("total KG nodes:", len(knowledge_graph.nodes))
            print("candidate CHUNK nodes:", len(candidate_nodes))
            print("skipped:", dict(skipped))

            entity_counts = [
                len(self._clean_terms(node.properties.get(self.property_name, [])))
                for node in candidate_nodes
            ]
            if entity_counts:
                print("entity count distribution:", dict(Counter(entity_counts)))
                print("min entities:", min(entity_counts))
                print("max entities:", max(entity_counts))
                print("avg entities:", round(sum(entity_counts) / len(entity_counts), 2))

        return candidate_nodes

    async def _generate_scenarios(
        self,
        n: int,
        knowledge_graph: KnowledgeGraph,
        persona_list: list[Persona],
        callbacks,
    ) -> list[SingleHopScenario]:

        if not persona_list:
            raise ValueError(
                "persona_list is empty. Generate personas first or pass a manual persona list."
            )

        nodes = self.get_node_clusters(knowledge_graph)

        if not nodes:
            raise ValueError(
                f"No valid CHUNK nodes found with `{self.property_name}`."
            )

        random.shuffle(nodes)

        samples_per_node = max(1, math.ceil(n / len(nodes)))
        scenarios = []
        skipped = Counter()

        for node in nodes:
            if len(scenarios) >= n:
                break

            terms = self._clean_terms(node.properties.get(self.property_name, []))

            if not terms:
                skipped["no_terms"] += 1
                continue

            prompt_input = ThemesPersonasInput(
                themes=terms,
                personas=persona_list,
            )

            try:
                persona_concepts = await self.theme_persona_matching_prompt.generate(
                    data=prompt_input,
                    llm=self.llm,
                    callbacks=callbacks,
                )
                persona_mapping = persona_concepts.mapping
            except Exception as e:
                skipped["persona_matching_failed"] += 1
                if self.debug:
                    print("[ragas-like-single-hop] persona matching failed:", repr(e))
                persona_mapping = {}

            persona_lookup = PersonaList(personas=persona_list)
            valid_personas = []

            for persona_name, concepts in persona_mapping.items():
                concept_l = [str(c).lower() for c in concepts]
                if any(term.lower() in concept_l for term in terms):
                    persona = persona_lookup[persona_name]
                    if persona:
                        valid_personas.append(persona)

            # Ragas normally depends on persona matching.
            # For robustness, keep all personas if matching returns empty.
            if not valid_personas:
                valid_personas = persona_list

            base_candidates = []

            for term in terms:
                for persona in valid_personas:
                    for style in QueryStyle:
                        for length in QueryLength:
                            base_candidates.append(
                                {
                                    "term": term,
                                    "node": node,
                                    "persona": persona,
                                    "style": style,
                                    "length": length,
                                }
                            )

            random.shuffle(base_candidates)

            selected = base_candidates[:samples_per_node]

            for item in selected:
                scenarios.append(
                    SingleHopScenario(
                        term=item["term"],
                        nodes=[item["node"]],
                        persona=item["persona"],
                        style=item["style"],
                        length=item["length"],
                    )
                )

                if len(scenarios) >= n:
                    break

        if self.debug:
            print("\n[ragas-like-single-hop] scenario stats")
            print("requested scenarios:", n)
            print("generated scenarios:", len(scenarios))
            print("skipped:", dict(skipped))
            print("style distribution:", dict(Counter(s.style.name for s in scenarios)))
            print("length distribution:", dict(Counter(s.length.name for s in scenarios)))
            print("persona distribution:", dict(Counter(s.persona.name for s in scenarios if s.persona)))
            print("sample terms:", [s.term for s in scenarios[:10]])

        return scenarios[:n]
    def make_contexts(self, scenario: SingleHopScenario) -> list[str]:
        return [
            make_single_hop_context(node)
            for node in scenario.nodes
        ]

    async def _generate_sample(self, scenario, callbacks):
        if not isinstance(scenario, SingleHopScenario):
            raise TypeError("scenario type should be SingleHopScenario")

        reference_contexts = self.make_contexts(scenario)
        reference_context = reference_contexts[0]

        prompt_input = QueryCondition(
            persona=scenario.persona,
            term=scenario.term,
            context=reference_context,
            query_length=scenario.length.value,
            query_style=scenario.style.value,
            llm_context=self.llm_context,
        )

        response = await self.generate_query_reference_prompt.generate(
            data=prompt_input,
            llm=self.llm,
            callbacks=callbacks,
        )

        return SingleTurnSample(
            user_input=response.query,
            reference=response.answer,
            reference_contexts=reference_contexts,
            persona_name=getattr(scenario.persona, "name", None),
            query_style=getattr(scenario.style, "name", None),
            query_length=getattr(scenario.length, "name", None),
        )

In [38]:
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.abstract import MultiHopAbstractQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.specific import MultiHopSpecificQuerySynthesizer

ragas_like_single_hop = RagasLikeSingleHopQuerySynthesizer(
    llm=generator.llm,
    llm_context=generator.llm_context,
)

ragas_like_single_hop.debug = True
# ragas_like_single_hop.max_entities_per_node = 30

query_distribution_single_hop = [
    (ragas_like_single_hop, 1.0),
]


In [39]:
# Generate QA from the cached KG.
# This does NOT rerun summaries, embeddings, themes, entities, or relationship builders.
testset = generator.generate(
    testset_size=300,
    query_distribution=query_distribution_single_hop,
    with_debugging_logs=True,
    raise_exceptions=True,
    debug=True,
)

[debug] generate: testset_size = 300
[debug] generate: query_distribution =
         ragas_like_single_hop_query_synthesizer 1.0
[debug] generate: using existing persona_list
[debug] generate: personas =
         name='Procurement Officer' role_description='Manages tender processes, ensures alignment with updates like TB-0012, and oversees communication through platforms such as B2B (ARIBA).'
         name='Offshore Energy Project Manager' role_description='Oversees the planning, execution, and management of large-scale offshore oil and gas projects, ensuring compliance with contract terms and operational performance standards.'
         name='EPC Project Tender Manager' role_description='Oversees tender submissions and manages project updates for Engineering, Procurement, and Construction contracts, ensuring compliance with ITT conditions and clarifications.'
[debug] generate: scenario splits = [300]
[debug] generate: submit scenario generation ragas_like_single_hop_query_synthesizer 

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]


[ragas-like-single-hop] candidate node stats
total KG nodes: 1555
candidate CHUNK nodes: 1553
skipped: {'missing_entities': 2}
entity count distribution: {11: 115, 6: 144, 8: 178, 5: 83, 7: 182, 9: 167, 12: 90, 13: 70, 2: 28, 10: 135, 4: 63, 3: 51, 16: 32, 14: 61, 17: 30, 1: 4, 15: 53, 30: 4, 23: 5, 20: 5, 25: 6, 26: 2, 19: 14, 18: 16, 22: 6, 21: 4, 24: 1, 28: 3, 27: 1}
min entities: 1
max entities: 30
avg entities: 9.44


Generating Scenarios: 100%|██████████| 1/1 [09:40<00:00, 580.08s/it]



[ragas-like-single-hop] scenario stats
requested scenarios: 300
generated scenarios: 300
skipped: {}
style distribution: {'WEB_SEARCH_LIKE': 74, 'MISSPELLED': 80, 'POOR_GRAMMAR': 87, 'PERFECT_GRAMMAR': 59}
length distribution: {'SHORT': 93, 'LONG': 106, 'MEDIUM': 101}
persona distribution: {'Offshore Energy Project Manager': 109, 'Procurement Officer': 99, 'EPC Project Tender Manager': 92}
sample terms: ['Legrand', 'E2s - Break Glass Button (Ex)', 'Ichthys Onshore AGRU and CCS Project', 'section 3.15 - Reporting & Communication', 'Primary Structure (Stick-build)', 'AS/NZS 4761.1', 'Vendor Equipment Manuals (VEM)', 'Power Distribution Control System', 'modern slavery statement(s)', 'Contract']
[debug] generate: scenario_sample_list sizes = [300]


Generating Samples: 100%|██████████| 300/300 [00:11<00:00, 26.60it/s] 

[debug] generate: final samples = 300


In [40]:
from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_singlehop_v3.jsonl")
rows = []
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for sample in testset.samples:
        ev = sample.eval_sample

        row = {
            "question": ev.user_input,
            "answer": ev.reference,
            "synthesizer_name": sample.synthesizer_name,
            "persona_name": getattr(ev, "persona_name", None),
            "query_style": getattr(ev, "query_style", None),
            "query_length": getattr(ev, "query_length", None),
            "reference_contexts": ev.reference_contexts,
        }
        rows.append(row)

        f.write(json.dumps(row,indent=2, ensure_ascii=False) + "\n")

print(f"Saved {len(testset.samples)} Q/A rows to {OUTPUT_PATH}")

Saved 300 Q/A rows to synthetic_qa_ragas_singlehop_v3.jsonl


In [ ]:
### read, load

from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_singlehop_v2.jsonl")

text = OUTPUT_PATH.read_text(encoding="utf-8").strip()

rows = []
decoder = json.JSONDecoder()
idx = 0

while idx < len(text):
    # skip whitespace/newlines between objects
    while idx < len(text) and text[idx].isspace():
        idx += 1

    if idx >= len(text):
        break

    obj, end = decoder.raw_decode(text, idx)
    rows.append(obj)
    idx = end

print("loaded rows:", len(rows))
print(rows[0].keys())

loaded rows: 300
dict_keys(['question', 'answer', 'synthesizer_name', 'persona_name', 'query_style', 'query_length', 'reference_contexts'])


In [113]:
SINGLEHOP_QA_JUDGE_PROMPT = """
You are a strict evaluator for synthetic single-hop QA data used to evaluate a tender/EPC RAG system.

Evaluate whether the question-answer pair is high-quality, faithful, and useful.

Use ONLY the provided reference context.

Scoring scale:
- 5 = excellent
- 4 = good
- 3 = acceptable but imperfect
- 2 = weak
- 1 = bad
- 0 = unusable

Metrics:

1. faithfulness
Does the answer contain only facts supported by the reference context?
Penalize invented obligations, wrong parties, wrong dates, wrong exhibit names, wrong clauses, or unsupported synthesis.

2. question_relevance
Is the question relevant to the reference context and tender/EPC domain?
Penalize generic, vague, unrelated, or impossible questions.

3. answer_relevance
Does the answer directly and completely answer the question?
Penalize evasive, incomplete, overly broad, or off-topic answers.

4. context_sufficiency
Does the reference context contain enough evidence to answer the question?
Penalize if key information is missing.

5. single_context_focus
Can the question reasonably be answered from this one context?
Score low if the question obviously requires other documents, cross-document reasoning, or broad aggregation outside this context.

6. specificity
Is the question specific enough to be useful for RAG evaluation?
Score low if it is too vague, trivial, or generic.

Return STRICT JSON only:
{
  "faithfulness": 0,
  "question_relevance": 0,
  "answer_relevance": 0,
  "context_sufficiency": 0,
  "single_context_focus": 0,
  "specificity": 0,
  "overall_score": 0,
  "verdict": "keep|review|drop",
  "is_valid_singlehop": true,
  "answer_supported": true,
  "unsupported_claims": [],
  "issues": ["..."],
  "explanation": "short explanation"
}
""".strip()

In [114]:
def parse_json_object(raw):
    raw = (raw or "").strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, flags=re.S)
    return json.loads(m.group(0) if m else raw)


In [115]:
async def judge_singlehop_quality(llm, row, max_context_chars: int = 7000):
    contexts = row.get("reference_contexts") or []

    # Single-hop usually has one context, but keep all if present.
    context = "\n\n---\n\n".join(str(c) for c in contexts)
    context = context[:max_context_chars]

    prompt = f"""
Question:
{row.get("question")}

Answer:
{row.get("answer")}

Reference context:
{context}
""".strip()

    result = await llm.ainvoke([
        SystemMessage(content=SINGLEHOP_QA_JUDGE_PROMPT),
        HumanMessage(content=prompt),
    ])

    data = parse_json_object(result.content)

    score_fields = [
        "faithfulness",
        "question_relevance",
        "answer_relevance",
        "context_sufficiency",
        "single_context_focus",
        "specificity",
    ]

    scores = [float(data.get(k, 0) or 0) for k in score_fields]

    if not data.get("overall_score"):
        data["overall_score"] = round(sum(scores) / len(scores), 2)

    if not data.get("verdict"):
        if (
            data["overall_score"] >= 4
            and data.get("faithfulness", 0) >= 4
            and data.get("context_sufficiency", 0) >= 4
            and data.get("single_context_focus", 0) >= 4
        ):
            data["verdict"] = "keep"
        elif data["overall_score"] >= 3:
            data["verdict"] = "review"
        else:
            data["verdict"] = "drop"

    return data

In [116]:
singlehop_rows = [
    r for r in rows
    if "single_hop" in (r.get("synthesizer_name") or "")
]

print("singlehop rows:", len(singlehop_rows))

singlehop_judgements = []

for i, row in enumerate(singlehop_rows, start=1):
    print(f"[singlehop judge] {i}/{len(singlehop_rows)}")
    j = await judge_singlehop_quality(llm, row)
    singlehop_judgements.append(j)

print("judgements:", len(singlehop_judgements))

singlehop rows: 300
[singlehop judge] 1/300
[singlehop judge] 2/300
[singlehop judge] 3/300
[singlehop judge] 4/300
[singlehop judge] 5/300
[singlehop judge] 6/300
[singlehop judge] 7/300
[singlehop judge] 8/300
[singlehop judge] 9/300
[singlehop judge] 10/300
[singlehop judge] 11/300
[singlehop judge] 12/300
[singlehop judge] 13/300
[singlehop judge] 14/300
[singlehop judge] 15/300
[singlehop judge] 16/300
[singlehop judge] 17/300
[singlehop judge] 18/300
[singlehop judge] 19/300
[singlehop judge] 20/300
[singlehop judge] 21/300
[singlehop judge] 22/300
[singlehop judge] 23/300
[singlehop judge] 24/300
[singlehop judge] 25/300
[singlehop judge] 26/300
[singlehop judge] 27/300
[singlehop judge] 28/300
[singlehop judge] 29/300
[singlehop judge] 30/300
[singlehop judge] 31/300
[singlehop judge] 32/300
[singlehop judge] 33/300
[singlehop judge] 34/300
[singlehop judge] 35/300
[singlehop judge] 36/300
[singlehop judge] 37/300
[singlehop judge] 38/300
[singlehop judge] 39/300
[singlehop jud

In [117]:
judged_singlehop_rows = []

for row, judgment in zip(singlehop_rows, singlehop_judgements):
    judged_singlehop_rows.append({
        **row,
        "judge": judgment,
        "judge_overall_score": judgment.get("overall_score"),
        "judge_verdict": judgment.get("verdict"),
        "judge_is_valid_singlehop": judgment.get("is_valid_singlehop"),
    })

print("judged rows:", len(judged_singlehop_rows))

judged rows: 300


In [119]:
judged_singlehop_rows

[{'question': 'Which category does Legrand belong to in the approved vendor list for the Ichthys Onshore AGRU Upgrade and CCS Project?',
  'answer': "Legrand is listed under the category 'Control Panel Enclosures – Relays - Indoor' in the approved vendor list for the Ichthys Onshore AGRU Upgrade and CCS Project.",
  'synthesizer_name': 'ragas_like_single_hop_query_synthesizer',
  'persona_name': 'Offshore Energy Project Manager',
  'query_style': 'WEB_SEARCH_LIKE',
  'query_length': 'SHORT',
  'reference_contexts': ['chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_11_Exhibit_I_Major_Subcontractors_and_Vendors_Rev_0_20241216_pdf_size_541323_page_10_text_chunk_1\nfile_name: ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 11_Exhibit I_Major Subcontractors and Vendors_Rev.0_20241216.pdf\npage_number: 10\n\nTender No: WS33071728 \nICHTHYS ONSHORE AGRU UPGRADE AND CCS PROJECT EPC CONTRACT \nEXHIBIT I – 

In [118]:
kept_singlehop_rows = [
    row for row in judged_singlehop_rows
    if row.get("judge_verdict") == "keep"
    and row.get("judge_is_valid_singlehop") is True
    and float(row["judge"].get("faithfulness", 0) or 0) >= 4
    and float(row["judge"].get("context_sufficiency", 0) or 0) >= 4
    and float(row["judge"].get("single_context_focus", 0) or 0) >= 4
    and float(row["judge"].get("answer_relevance", 0) or 0) >= 4
]

print("kept singlehop rows:", len(kept_singlehop_rows))

kept singlehop rows: 265


In [120]:
OUTPUT_JUDGED_PATH = Path("synthetic_qa_ragas_singlehop_v3_judged.jsonl")
OUTPUT_KEEP_PATH = Path("synthetic_qa_ragas_singlehop_v3_kept.jsonl")

with OUTPUT_JUDGED_PATH.open("w", encoding="utf-8") as f:
    for row in judged_singlehop_rows:
        f.write(json.dumps(row, ensure_ascii=False, indent= 2) + "\n")

with OUTPUT_KEEP_PATH.open("w", encoding="utf-8") as f:
    for row in kept_singlehop_rows:
        f.write(json.dumps(row, ensure_ascii=False,indent =2) + "\n")

print(f"saved {len(judged_singlehop_rows)} judged rows to {OUTPUT_JUDGED_PATH}")
print(f"saved {len(kept_singlehop_rows)} kept rows to {OUTPUT_KEEP_PATH}")

saved 300 judged rows to synthetic_qa_ragas_singlehop_v3_judged.jsonl
saved 265 kept rows to synthetic_qa_ragas_singlehop_v3_kept.jsonl


### Multihop-query

In [41]:
import random
import numpy as np

from ragas.testset.synthesizers.base import QueryStyle, QueryLength
from ragas.testset.synthesizers.multi_hop.abstract import MultiHopAbstractQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.specific import MultiHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.base import MultiHopScenario
from ragas.testset.synthesizers.multi_hop.prompts import ConceptsList
from ragas.testset.synthesizers.prompts import ThemesPersonasInput
from ragas.testset.persona import PersonaList

In [42]:
from collections import Counter
import random
import numpy as np

from ragas.testset.synthesizers.base import QueryStyle, QueryLength
from ragas.testset.synthesizers.multi_hop.abstract import MultiHopAbstractQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.base import MultiHopScenario
from ragas.testset.synthesizers.multi_hop.prompts import ConceptsList
from ragas.testset.synthesizers.prompts import ThemesPersonasInput
from ragas.testset.persona import PersonaList
from ragas import SingleTurnSample
from ragas.testset.synthesizers.multi_hop.base import MultiHopScenario
from ragas.testset.synthesizers.multi_hop.prompts import QueryConditions
from ragas.testset.synthesizers.base import Scenario

def make_contexts(self, scenario: MultiHopScenario) -> list[str]:
    contexts = []

    for i, node in enumerate(scenario.nodes):
        metadata = node.properties.get("document_metadata", {}) or {}

        chunk_id = metadata.get("chunk_id", "")
        file_name = metadata.get("file_name", "")
        page_number = metadata.get("page_number", "")

        page_content = node.properties.get("page_content", "")

        context = f"""<{i + 1}-hop>
chunk_id: {chunk_id}
file_name: {file_name}
page_number: {page_number}

{page_content}"""

        contexts.append(context)

    return contexts




class StrictRagasLikeMultiHopAbstractQuerySynthesizer(MultiHopAbstractQuerySynthesizer):
    name: str = "strict_ragas_like_multi_hop_abstract_query_synthesizer"
    max_hops: int = 3
    scenarios_per_combo: int = 2
    debug: bool = True

    async def _generate_scenarios(
        self,
        n: int,
        knowledge_graph,
        persona_list,
        callbacks,
    ):
        node_clusters = self.get_node_clusters(knowledge_graph, n=n)

        if self.debug:
            cluster_sizes = [len(c) for c in node_clusters]
            print("\n[strict-abstract] requested scenarios:", n)
            print("[strict-abstract] clusters found:", len(node_clusters))
            print("[strict-abstract] cluster size distribution:", dict(Counter(cluster_sizes)))
            if cluster_sizes:
                print("[strict-abstract] min cluster size:", min(cluster_sizes))
                print("[strict-abstract] max cluster size:", max(cluster_sizes))
                print("[strict-abstract] avg cluster size:", round(sum(cluster_sizes) / len(cluster_sizes), 2))

        if not node_clusters:
            raise ValueError("No summary_similarity clusters found.")

        scenarios = []
        skipped = Counter()
        persona_counter = Counter()
        style_counter = Counter()
        length_counter = Counter()
        hop_counter = Counter()
        combo_size_counter = Counter()

        random.shuffle(node_clusters)

        for cluster_idx, cluster in enumerate(node_clusters):
            if len(scenarios) >= n:
                break

            nodes = list(cluster)

            if len(nodes) < 2:
                skipped["cluster_less_than_2_nodes"] += 1
                continue

            nodes = nodes[: self.max_hops]

            node_themes = [
                node.properties.get(self.abstract_property_name, [])
                for node in nodes
            ]

            if sum(bool(x) for x in node_themes) < 2:
                skipped["less_than_2_nodes_with_themes"] += 1
                continue

            concept_input = ConceptsList(
                lists_of_concepts=node_themes,
                max_combinations=3,
            )

            try:
                concept_combination = await self.concept_combination_prompt.generate(
                    data=concept_input,
                    llm=self.llm,
                    callbacks=callbacks,
                )
            except Exception as e:
                skipped[f"concept_combination_error:{type(e).__name__}"] += 1
                continue

            combinations = [
                combo for combo in concept_combination.combinations
                if isinstance(combo, list) and len(combo) >= 2
            ]

            if not combinations:
                skipped["no_valid_concept_combinations"] += 1
                continue

            flattened_themes = [
                theme
                for combo in combinations
                for theme in combo
            ]

            persona_input = ThemesPersonasInput(
                themes=flattened_themes,
                personas=persona_list,
            )

            try:
                persona_concepts = await self.theme_persona_matching_prompt.generate(
                    data=persona_input,
                    llm=self.llm,
                    callbacks=callbacks,
                )
            except Exception as e:
                skipped[f"persona_matching_error:{type(e).__name__}"] += 1
                continue

            persona_lookup = PersonaList(personas=persona_list)

            for combo in combinations:
                if len(scenarios) >= n:
                    break

                valid_personas = []

                for persona_name, concept_list in persona_concepts.mapping.items():
                    concept_list = [c.lower() for c in concept_list]
                    if any(c.lower() in concept_list for c in combo):
                        persona = persona_lookup[persona_name]
                        if persona:
                            valid_personas.append(persona)

                if not valid_personas:
                    valid_personas = persona_list

                candidates = []

                for persona in valid_personas:
                    for style in QueryStyle:
                        for length in QueryLength:
                            candidates.append(
                                MultiHopScenario(
                                    nodes=nodes,
                                    combinations=combo,
                                    persona=persona,
                                    style=style,
                                    length=length,
                                )
                            )

                if not candidates:
                    skipped["no_candidates"] += 1
                    continue

                random.shuffle(candidates)
                selected = candidates[: self.scenarios_per_combo]
                scenarios.extend(selected)

                for scenario in selected:
                    persona_counter[getattr(scenario.persona, "name", None)] += 1
                    style_counter[getattr(scenario.style, "name", None)] += 1
                    length_counter[getattr(scenario.length, "name", None)] += 1
                    hop_counter[len(scenario.nodes)] += 1
                    combo_size_counter[len(scenario.combinations)] += 1

        scenarios = scenarios[:n]

        if self.debug:
            print("[strict-abstract] scenarios generated:", len(scenarios))
            print("[strict-abstract] skipped:", dict(skipped))
            print("[strict-abstract] scenario hop distribution:", dict(hop_counter))
            print("[strict-abstract] concept-combo size distribution:", dict(combo_size_counter))
            print("[strict-abstract] persona distribution:", dict(persona_counter))
            print("[strict-abstract] style distribution:", dict(style_counter))
            print("[strict-abstract] length distribution:", dict(length_counter))

        return scenarios
    async def _generate_sample(self, scenario: Scenario, callbacks):
        if not isinstance(scenario, MultiHopScenario):
            raise TypeError("scenario type should be MultiHopScenario")

        reference_context = self.make_contexts(scenario)

        prompt_input = QueryConditions(
            persona=scenario.persona,
            themes=scenario.combinations,
            context=reference_context,
            query_length=scenario.length.value,
            query_style=scenario.style.value,
            llm_context=self.llm_context,
        )

        response = await self.generate_query_reference_prompt.generate(
            data=prompt_input,
            llm=self.llm,
            callbacks=callbacks,
        )

        return SingleTurnSample(
            user_input=response.query,
            reference=response.answer,
            reference_contexts=reference_context,
            persona_name=getattr(scenario.persona, "name", None),
            query_style=getattr(scenario.style, "name", None),
            query_length=getattr(scenario.length, "name", None),
        )
        
    def make_contexts(self, scenario: MultiHopScenario) -> list[str]:
        contexts = []

        for i, node in enumerate(scenario.nodes):
            metadata = node.properties.get("document_metadata", {}) or {}

            chunk_id = metadata.get("chunk_id", "")
            file_name = metadata.get("file_name", "")
            page_number = metadata.get("page_number", "")

            page_content = node.properties.get("page_content", "")

            context = f"""<{i + 1}-hop>
chunk_id: {chunk_id}
file_name: {file_name}
page_number: {page_number}

{page_content}"""

            contexts.append(context)

        return contexts

In [43]:
from ragas.testset.persona import generate_personas_from_kg

if generator.persona_list is None:
    generator.persona_list = generate_personas_from_kg(
        llm=generator.llm,
        kg=generator.knowledge_graph,
        num_personas=3,
        callbacks=[],
    )

print("personas:", len(generator.persona_list))
for p in generator.persona_list:
    print("-", p.name, ":", p.role_description)

personas: 3
- Procurement Officer : Manages tender processes, ensures alignment with updates like TB-0012, and oversees communication through platforms such as B2B (ARIBA).
- Offshore Energy Project Manager : Oversees the planning, execution, and management of large-scale offshore oil and gas projects, ensuring compliance with contract terms and operational performance standards.
- EPC Project Tender Manager : Oversees tender submissions and manages project updates for Engineering, Procurement, and Construction contracts, ensuring compliance with ITT conditions and clarifications.


In [ ]:
# Test only scenario generation, not final QA generation

strict_abstract = StrictRagasLikeMultiHopAbstractQuerySynthesizer(
    llm=generator.llm,
    llm_context=generator.llm_context,
)

strict_abstract.max_hops = 3
strict_abstract.scenarios_per_combo = 2
strict_abstract.debug = True

scenarios = await strict_abstract._generate_scenarios(
    n=10,
    knowledge_graph=generator.knowledge_graph,
    persona_list=generator.persona_list,
    callbacks=[],
)
print("scenarios:", len(scenarios))

for i, s in enumerate(scenarios[:5], start=1):
    print("\n--- scenario", i, "---")
    print("nodes:", len(s.nodes))
    print("combinations:", s.combinations)
    print("persona:", getattr(s.persona, "name", None))
    print("style:", getattr(s.style, "name", None), "|", getattr(s.style, "value", None))
    print("length:", getattr(s.length, "name", None), "|", getattr(s.length, "value", None))

    for j, node in enumerate(s.nodes, start=1):
        md = node.properties.get("document_metadata", {}) or {}
        print(f"  hop {j}:")
        print("    chunk_id:", md.get("chunk_id"))
        print("    file_name:", md.get("file_name"))
        print("    page_number:", md.get("page_number"))
        print("    themes:", node.properties.get("themes", [])[:5])


[strict-abstract] requested scenarios: 10
[strict-abstract] clusters found: 10
[strict-abstract] cluster size distribution: {3: 10}
[strict-abstract] min cluster size: 3
[strict-abstract] max cluster size: 3
[strict-abstract] avg cluster size: 3.0
[strict-abstract] scenarios generated: 10
[strict-abstract] skipped: {}
[strict-abstract] scenario hop distribution: {3: 10}
[strict-abstract] concept-combo size distribution: {2: 6, 3: 4}
[strict-abstract] persona distribution: {'EPC Project Manager': 4, 'Procurement Coordinator': 5, 'Energy Sector Analyst': 1}
[strict-abstract] style distribution: {'PERFECT_GRAMMAR': 4, 'POOR_GRAMMAR': 4, 'MISSPELLED': 2}
[strict-abstract] length distribution: {'LONG': 3, 'SHORT': 3, 'MEDIUM': 4}
scenarios: 10

--- scenario 1 ---
nodes: 3
combinations: ['Mechanical rotating equipment', 'EPC contract']
persona: EPC Project Manager
style: PERFECT_GRAMMAR | Perfect grammar
length: LONG | long
  hop 1:
    chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_a

In [44]:
class StrictRagasLikeMultiHopSpecificQuerySynthesizer(MultiHopSpecificQuerySynthesizer):
    name: str = "strict_ragas_like_multi_hop_specific_query_synthesizer"

    async def _generate_scenarios(
        self,
        n: int,
        knowledge_graph,
        persona_list,
        callbacks,
    ):
        triplets = self.get_node_clusters(knowledge_graph)

        if not triplets:
            raise ValueError("No entities_overlap relationships found.")

        scenarios = []
        random.shuffle(triplets)
        persona_lookup = PersonaList(personas=persona_list)

        for triplet in triplets:
            if len(scenarios) >= n:
                break

            node_a, rel, node_b = triplet[0], triplet[1], triplet[2]

            if node_a == node_b:
                continue

            overlapped_items = rel.properties.get(self.relation_overlap_property, [])
            themes = self._extract_themes_from_overlaps(overlapped_items)

            if not themes:
                continue

            persona_input = ThemesPersonasInput(
                themes=themes,
                personas=persona_list,
            )

            persona_concepts = await self.theme_persona_matching_prompt.generate(
                data=persona_input,
                llm=self.llm,
                callbacks=callbacks,
            )

            valid_personas = []

            for persona_name, concept_list in persona_concepts.mapping.items():
                persona = persona_lookup[persona_name]
                if persona:
                    valid_personas.append(persona)

            if not valid_personas:
                valid_personas = persona_list

            candidates = []

            for theme in themes:
                for persona in valid_personas:
                    for style in QueryStyle:
                        for length in QueryLength:
                            candidates.append(
                                MultiHopScenario(
                                    nodes=[node_a, node_b],
                                    combinations=[theme],
                                    persona=persona,
                                    style=style,
                                    length=length,
                                )
                            )

            random.shuffle(candidates)
            scenarios.extend(candidates[:2])

        return scenarios[:n]
    async def _generate_sample(self, scenario: Scenario, callbacks):
        if not isinstance(scenario, MultiHopScenario):
            raise TypeError("scenario type should be MultiHopScenario")

        reference_context = self.make_contexts(scenario)

        prompt_input = QueryConditions(
            persona=scenario.persona,
            themes=scenario.combinations,
            context=reference_context,
            query_length=scenario.length.value,
            query_style=scenario.style.value,
            llm_context=self.llm_context,
        )

        response = await self.generate_query_reference_prompt.generate(
            data=prompt_input,
            llm=self.llm,
            callbacks=callbacks,
        )

        return SingleTurnSample(
            user_input=response.query,
            reference=response.answer,
            reference_contexts=reference_context,
            persona_name=getattr(scenario.persona, "name", None),
            query_style=getattr(scenario.style, "name", None),
            query_length=getattr(scenario.length, "name", None),
        )
    def make_contexts(self, scenario: MultiHopScenario) -> list[str]:
        contexts = []

        for i, node in enumerate(scenario.nodes):
            metadata = node.properties.get("document_metadata", {}) or {}

            chunk_id = metadata.get("chunk_id", "")
            file_name = metadata.get("file_name", "")
            page_number = metadata.get("page_number", "")

            page_content = node.properties.get("page_content", "")

            context = f"""<{i + 1}-hop>
            chunk_id: {chunk_id}
            file_name: {file_name}
            page_number: {page_number}

            {page_content}"""

            contexts.append(context)

        return contexts

In [45]:
query_distribution_strict_multihop = [
    (
        StrictRagasLikeMultiHopAbstractQuerySynthesizer(
            llm=generator.llm,
            llm_context=generator.llm_context,
        ),
        0.5,
    ),
    (
        StrictRagasLikeMultiHopSpecificQuerySynthesizer(
            llm=generator.llm,
            llm_context=generator.llm_context,
        ),
        0.5,
    ),
]

In [46]:
testset_multihop = generator.generate(
    testset_size=100,
    query_distribution=query_distribution_strict_multihop,
    with_debugging_logs=True,
    raise_exceptions=False,
    debug=True,
)

[debug] generate: testset_size = 100
[debug] generate: query_distribution =
         multi_hop_abstract_query_synthesizer 0.5
         multi_hop_specific_query_synthesizer 0.5
[debug] generate: using existing persona_list
[debug] generate: personas =
         name='Procurement Officer' role_description='Manages tender processes, ensures alignment with updates like TB-0012, and oversees communication through platforms such as B2B (ARIBA).'
         name='EPC Project Tender Manager' role_description='Oversees tender submissions and manages project updates for Engineering, Procurement, and Construction contracts, ensuring compliance with ITT conditions and clarifications.'
         name='Offshore Energy Project Manager' role_description='Oversees the planning, execution, and management of large-scale offshore oil and gas projects, ensuring compliance with contract terms and operational performance standards.'
[debug] generate: scenario splits = [50, 50]
[debug] generate: submit scenario g

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]


[strict-abstract] requested scenarios: 50
[strict-abstract] clusters found: 50
[strict-abstract] cluster size distribution: {3: 50}
[strict-abstract] min cluster size: 3
[strict-abstract] max cluster size: 3
[strict-abstract] avg cluster size: 3.0


Generating Scenarios:  50%|█████     | 1/2 [00:17<00:17, 17.90s/it]

[strict-abstract] scenarios generated: 50
[strict-abstract] skipped: {}
[strict-abstract] scenario hop distribution: {3: 50}
[strict-abstract] concept-combo size distribution: {2: 50}
[strict-abstract] persona distribution: {'Procurement Officer': 9, 'EPC Project Tender Manager': 16, 'Offshore Energy Project Manager': 25}
[strict-abstract] style distribution: {'WEB_SEARCH_LIKE': 16, 'POOR_GRAMMAR': 10, 'PERFECT_GRAMMAR': 10, 'MISSPELLED': 14}
[strict-abstract] length distribution: {'MEDIUM': 20, 'SHORT': 14, 'LONG': 16}


Generating Scenarios: 100%|██████████| 2/2 [00:46<00:00, 23.03s/it]


[debug] generate: scenario_sample_list sizes = [50, 50]


Generating Samples: 100%|██████████| 100/100 [00:15<00:00,  6.58it/s]

[debug] generate: final samples = 100


In [ ]:
from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_multihop_v2.jsonl")
rows = []
with OUTPUT_PATH.open("", encoding="utf-8") as f:
    for sample in testset_multihop.samples:
        ev = sample.eval_sample

        row = {
            "question": ev.user_input,
            "answer": ev.reference,
            "synthesizer_name": sample.synthesizer_name,
            "persona_name": getattr(ev, "persona_name", None),
            "query_style": getattr(ev, "query_style", None),
            "query_length": getattr(ev, "query_length", None),
            "reference_contexts": ev.reference_contexts,
        }
        rows.append(row)

        f.write(json.dumps(row,indent=2, ensure_ascii=False) + "\n")

print(f"Saved {len(testset_multihop.samples)} Q/A rows to {OUTPUT_PATH}")

Saved 100 Q/A rows to synthetic_qa_ragas_multihop_v2.jsonl


In [100]:
### read

from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_multihop_v1.jsonl")

text = OUTPUT_PATH.read_text(encoding="utf-8").strip()

rows = []
decoder = json.JSONDecoder()
idx = 0

while idx < len(text):
    # skip whitespace/newlines between objects
    while idx < len(text) and text[idx].isspace():
        idx += 1

    if idx >= len(text):
        break

    obj, end = decoder.raw_decode(text, idx)
    rows.append(obj)
    idx = end

print("loaded rows:", len(rows))
print(rows[0].keys())

loaded rows: 50
dict_keys(['question', 'answer', 'synthesizer_name', 'persona_name', 'query_style', 'query_length', 'reference_contexts'])


In [101]:
import re
from collections import Counter, defaultdict

def hop_count(reference_contexts):
    text = "\n".join(reference_contexts or [])
    return len(re.findall(r"<\d+-hop>", text))

def has_chunk_id(ctx):
    return "chunk_id:" in ctx

def structural_report(rows):
    report = {}

    report["total"] = len(rows)
    report["missing_question"] = sum(not r.get("question") for r in rows)
    report["missing_answer"] = sum(not r.get("answer") for r in rows)
    report["missing_reference_contexts"] = sum(not r.get("reference_contexts") for r in rows)

    report["synthesizer_distribution"] = Counter(
        r.get("synthesizer_name") for r in rows
    )

    report["hop_distribution"] = Counter(
        hop_count(r.get("reference_contexts")) for r in rows
    )

    report["query_style_distribution"] = Counter(
        r.get("query_style") for r in rows
    )

    report["query_length_distribution"] = Counter(
        r.get("query_length") for r in rows
    )

    report["contexts_with_chunk_id"] = sum(
        all(has_chunk_id(ctx) for ctx in (r.get("reference_contexts") or []))
        for r in rows
        if r.get("reference_contexts")
    )

    return report

report = structural_report(rows)
report

{'total': 50,
 'missing_question': 0,
 'missing_answer': 0,
 'missing_reference_contexts': 0,
 'synthesizer_distribution': Counter({'multi_hop_abstract_query_synthesizer': 25,
          'multi_hop_specific_query_synthesizer': 25}),
 'hop_distribution': Counter({2: 31, 3: 19}),
 'query_style_distribution': Counter({'PERFECT_GRAMMAR': 19,
          'POOR_GRAMMAR': 12,
          'MISSPELLED': 11,
          'WEB_SEARCH_LIKE': 8}),
 'query_length_distribution': Counter({'LONG': 24, 'SHORT': 16, 'MEDIUM': 10}),
 'contexts_with_chunk_id': 50}

In [102]:
MULTIHOP_QA_JUDGE_PROMPT = """
You are a strict evaluator for synthetic multi-hop QA data used to evaluate a tender/EPC RAG system.

Evaluate whether the question-answer pair is high-quality, faithful, and genuinely multi-hop.

Use ONLY the provided reference contexts.

Scoring scale:
- 5 = excellent
- 4 = good
- 3 = acceptable but imperfect
- 2 = weak
- 1 = bad
- 0 = unusable

Metrics:

1. faithfulness
Does the answer contain only facts supported by the reference contexts?
Penalize invented obligations, wrong parties, wrong dates, wrong exhibit names, wrong clauses, or unsupported synthesis.

2. question_relevance
Is the question relevant to the reference contexts and tender/EPC domain?
Penalize generic, vague, unrelated, or impossible questions.

3. answer_relevance
Does the answer directly and completely answer the question?
Penalize evasive, incomplete, overly broad, or off-topic answers.

4. context_sufficiency
Do the provided reference contexts contain enough evidence to answer the question?
Penalize if key information is missing.

5. multi_context_usage
Does the answer actually use information from at least two different reference contexts?
Score low if the answer can be fully answered from one context only.

6. reasoning_integration
Does the answer connect, compare, combine, or synthesize facts across contexts?
Score low if it merely copies facts without explaining the relationship.

7. context_specificity
Are the required contexts specific and meaningful, not just duplicated or near-identical text?
Score low if contexts are redundant duplicates.

Return STRICT JSON only:
{
  "faithfulness": 0,
  "question_relevance": 0,
  "answer_relevance": 0,
  "context_sufficiency": 0,
  "multi_context_usage": 0,
  "reasoning_integration": 0,
  "context_specificity": 0,
  "overall_score": 0,
  "verdict": "keep|review|drop",
  "is_genuine_multihop": true,
  "answer_supported": true,
  "uses_multiple_contexts": true,
  "required_context_numbers": [1, 2],
  "missing_context_numbers": [],
  "unsupported_claims": [],
  "issues": ["..."],
  "explanation": "short explanation"
}
""".strip()

In [103]:
from langchain_core.messages import SystemMessage, HumanMessage
import json
import re

def parse_json_object(raw):
    raw = (raw or "").strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, flags=re.S)
    return json.loads(m.group(0) if m else raw)

async def judge_multihop_quality(llm, row, max_context_chars_per_context: int = 5000):
    contexts = row.get("reference_contexts") or []

    numbered_contexts = []

    for i, context in enumerate(contexts, start=1):
        numbered_contexts.append(
            f"[Context {i}]\n{str(context)[:max_context_chars_per_context]}"
        )

    prompt = f"""
Question:
{row.get("question")}

Answer:
{row.get("answer")}

Reference contexts:
{chr(10).join(numbered_contexts)}
""".strip()

    result = await llm.ainvoke([
        SystemMessage(content=MULTIHOP_QA_JUDGE_PROMPT),
        HumanMessage(content=prompt),
    ])

    data = parse_json_object(result.content)

    score_fields = [
        "faithfulness",
        "question_relevance",
        "answer_relevance",
        "context_sufficiency",
        "multi_context_usage",
        "reasoning_integration",
        "context_specificity",
    ]

    scores = [float(data.get(k, 0) or 0) for k in score_fields]

    if not data.get("overall_score"):
        data["overall_score"] = round(sum(scores) / len(scores), 2)

    if not data.get("verdict"):
        if (
            data["overall_score"] >= 4
            and data.get("faithfulness", 0) >= 4
            and data.get("context_sufficiency", 0) >= 4
            and data.get("multi_context_usage", 0) >= 4
        ):
            data["verdict"] = "keep"
        elif data["overall_score"] >= 3:
            data["verdict"] = "review"
        else:
            data["verdict"] = "drop"

    return data

In [104]:
multihop_rows = [
    r for r in rows
    if "multi_hop" in (r.get("synthesizer_name") or "")
]

judgements = []

for row in multihop_rows:
    j = await judge_multihop_quality(llm, row)
    judgements.append({**j, "question": row["question"]})

judgements

[{'faithfulness': 4,
  'question_relevance': 5,
  'answer_relevance': 4,
  'context_sufficiency': 4,
  'multi_context_usage': 4,
  'reasoning_integration': 4,
  'context_specificity': 5,
  'overall_score': 30,
  'verdict': 'keep',
  'is_genuine_multihop': True,
  'answer_supported': True,
  'uses_multiple_contexts': True,
  'required_context_numbers': [1, 2],
  'missing_context_numbers': [],
  'unsupported_claims': [],
  'issues': [],
  'explanation': 'The answer is mostly faithful to the provided contexts, integrates information from multiple contexts, and demonstrates multi-hop reasoning. However, some details about the implications for operational testing could be more explicitly tied to the Approved Software List.',
  'question': 'How does the system testing and validation process outlined in the ICHTHYS Onshore AGRU Upgrade and CCS Project EPC Contract integrate with the Approved Software List, and what are the implications for operational testing?'},
 {'faithfulness': 5,
  'quest

In [105]:
# Attach judgments back to original multihop rows
judged_multihop_rows = []

for row, judgment in zip(multihop_rows, judgements):
    judged_multihop_rows.append({
        **row,
        "judge": judgment,
        "judge_overall_score": judgment.get("overall_score"),
        "judge_verdict": judgment.get("verdict"),
        "judge_is_genuine_multihop": judgment.get("is_genuine_multihop"),
    })

print("judged rows:", len(judged_multihop_rows))

judged rows: 50


In [106]:
kept_multihop_rows = [
    row for row in judged_multihop_rows
    if row.get("judge_verdict") == "keep"
]

print("kept rows:", len(kept_multihop_rows))

kept rows: 39


In [107]:
kept_multihop_rows = [
    row for row in judged_multihop_rows
    if row.get("judge_verdict") == "keep"
    and row.get("judge_is_genuine_multihop") is True
    and float(row["judge"].get("faithfulness", 0) or 0) >= 4
    and float(row["judge"].get("context_sufficiency", 0) or 0) >= 4
    and float(row["judge"].get("multi_context_usage", 0) or 0) >= 4
    and float(row["judge"].get("reasoning_integration", 0) or 0) >= 4
]

print("strict kept rows:", len(kept_multihop_rows))

strict kept rows: 39


In [108]:
kept_multihop_rows

[{'question': 'How does the system testing and validation process outlined in the ICHTHYS Onshore AGRU Upgrade and CCS Project EPC Contract integrate with the Approved Software List, and what are the implications for operational testing?',
  'answer': 'The system testing and validation process for the ICHTHYS Onshore AGRU Upgrade and CCS Project EPC Contract involves several critical steps, including loop checks for all control loops, functional testing of control logic and sequences, functional safety checks in accordance with IEC 61511 or relevant standards, and operational testing to simulate real-world conditions. Additionally, system integration commissioning ensures synchronization and communication testing across integrated systems, extended loop testing, and operational testing under company supervision to minimize disruption. This process is closely tied to the Approved Software List, as outlined in the contractor documentation, which specifies the software tools and systems a

In [111]:
from pathlib import Path
import json

OUTPUT_KEEP_PATH = Path("synthetic_qa_ragas_multihop_v1_kept.jsonl")

with OUTPUT_KEEP_PATH.open("w", encoding="utf-8") as f:
    for row in kept_multihop_rows:
        f.write(json.dumps(row, ensure_ascii=False, indent= 2) + "\n")

print(f"saved {len(kept_multihop_rows)} kept rows to {OUTPUT_KEEP_PATH}")

saved 39 kept rows to synthetic_qa_ragas_multihop_v1_kept.jsonl


### agreggate validated answer

In [51]:
# ---------------------------------------------------------------------
# Tender aggregation/list/summary synthesizer using Leiden communities
# Ragas-inspired: scenario generation -> sample generation
# ---------------------------------------------------------------------

import random
import typing as t
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from pydantic import BaseModel, Field

from ragas.prompt import PydanticPrompt
from ragas.testset.graph import KnowledgeGraph, Node, Relationship
from ragas.testset.persona import Persona, PersonaList
from ragas.testset.synthesizers.base import (
    BaseSynthesizer,
    BaseScenario,
    QueryStyle,
    QueryLength,
)
from ragas.testset.synthesizers.prompts import (
    ThemesPersonasInput,
    ThemesPersonasMatchingPrompt,
)
from ragas.testset.synthesizers.testset_schema import SingleTurnSample

In [54]:
# ---------------------------------------------------------------------
# 1. Scenario schema
# ---------------------------------------------------------------------

class TenderAggregationScenario(BaseScenario):
    community_id: str
    task_type: str
    themes: list[str] = Field(default_factory=list)
    entities: list[str] = Field(default_factory=list)

    def __repr__(self) -> str:
        return (
            "TenderAggregationScenario("
            f"community_id={self.community_id}, "
            f"nodes={len(self.nodes)}, "
            f"task_type={self.task_type}, "
            f"themes={self.themes[:5]}, "
            f"entities={self.entities[:5]}, "
            f"persona={self.persona}, "
            f"style={self.style}, "
            f"length={self.length})"
        )

In [131]:
# ---------------------------------------------------------------------
# 2. Prompt schema
# ---------------------------------------------------------------------

class TenderAggregationInput(BaseModel):
    persona: Persona
    task_type: str
    themes: list[str]
    entities: list[str]
    query_style: str
    query_length: str
    context: str
    llm_context: t.Optional[str] = None


class TenderAggregationOutput(BaseModel):
    query: str = Field(description="The generated aggregation/list/summary question.")
    answer: str = Field(description="The reference answer, fully supported by the context.")


class TenderAggregationPrompt(PydanticPrompt[TenderAggregationInput, TenderAggregationOutput]):
    instruction: str = """
Generate ONE high-quality aggregation/list/summary query and answer for a tender/EPC RAG evaluation dataset.

The query must require collecting, listing, or summarizing multiple facts from the provided context.
It should be answerable using ONLY the provided context.

Use the scenario conditions:
- persona: ask from this user's perspective
- task_type: controls the kind of aggregation question
- themes/entities: use them as anchors when relevant
- query_style: controls wording style
- query_length: controls question length

Task type meanings:
- list_requirements: ask for multiple requirements
- summarize_obligations: ask for obligations/responsibilities
- list_deliverables: ask for deliverables, documents, plans, reports, drawings, registers, certificates
- summarize_approval_requirements: ask about review, approval, acceptance, endorsement, notification
- summarize_contractor_responsibilities: ask about Contractor/Tenderer responsibilities
- list_documents_or_procedures: ask for documents/procedures listed in the context
- summarize_technical_requirements: ask about technical systems/equipment/specifications
- summarize_commercial_requirements: ask about payment, bonds, insurance, penalties, amounts, compensation
- summarize_hses_requirements: ask about HSES/security/environment/site safety obligations
- overview_scope: ask for an overview of the scope/purpose/topic represented by the context

Rules:
1. The answer must be faithful to the context.
2. Do not invent facts.
3. Do not use external knowledge.
4. Prefer tender-specific wording: Company, Contractor, Tenderer, Exhibit, clause, deliverable, approval, shall, must.
5. If the context does not contain enough information for the task_type, generate a narrower question that is still aggregation/list/summary.
6. Return only the query and answer in the required schema.
7. Do not mention the evaluation setup. The query must not contain phrases such as
   "provided context", "reference context", "given text", "above text",
   "source text", "excerpt", or "documents provided".
   Ask as a real tender user would ask.
""".strip()

    input_model: t.Type[TenderAggregationInput] = TenderAggregationInput
    output_model: t.Type[TenderAggregationOutput] = TenderAggregationOutput

In [132]:
# ---------------------------------------------------------------------
# 3. Helper lists
# ---------------------------------------------------------------------

TENDER_AGGREGATION_TASK_TYPES = [
    "list_requirements",
    "summarize_obligations",
    "list_deliverables",
    "summarize_approval_requirements",
    "summarize_contractor_responsibilities",
    "list_documents_or_procedures",
    "summarize_technical_requirements",
    "summarize_commercial_requirements",
    "summarize_hses_requirements",
    "overview_scope",
]

In [133]:
min_summary_similarity: float = 0.82
min_entity_overlap: float = 0.12
use_summary_edges: bool = True
use_entity_edges: bool = False

In [134]:
# ---------------------------------------------------------------------
# 4. Leiden-based aggregation synthesizer
# ---------------------------------------------------------------------

@dataclass
class LeidenTenderAggregationQuerySynthesizer(BaseSynthesizer[TenderAggregationScenario]):
    name: str = "leiden_tender_aggregation_query_synthesizer"

    # Ragas-style LLM persona matching prompt
    theme_persona_matching_prompt: PydanticPrompt = field(
        default_factory=ThemesPersonasMatchingPrompt
    )

    # Final QA prompt
    generate_query_reference_prompt: PydanticPrompt = field(
        default_factory=TenderAggregationPrompt
    )

    # Which Ragas KG relationship types to use for Leiden
    relationship_types: tuple[str, ...] = (
        "summary_similarity",
        "entities_overlap",
    )

    min_community_size: int = 2
    max_nodes_per_context: int = 10
    max_context_chars_per_node: int = 2500
    max_terms: int = 20
    debug: bool = True

    def _relationship_is_usable(self, rel: Relationship) -> bool:
        summary_score = rel.get_property("summary_similarity")
        entity_score = (
            rel.get_property("entities_overlap_score")
            or rel.get_property("entities_overlap_overlap_score")
        )

        if self.use_summary_edges and rel.type == "summary_similarity":
            return summary_score is not None and float(summary_score) >= self.min_summary_similarity

        if self.use_entity_edges and rel.type == "entities_overlap":
            return entity_score is not None and float(entity_score) >= self.min_entity_overlap

        return False

    def _relationship_weight(self, rel: Relationship) -> float:
        if rel.get_property("summary_similarity") is not None:
            return float(rel.get_property("summary_similarity") or 0.0)

        if rel.get_property("entities_overlap_score") is not None:
            return float(rel.get_property("entities_overlap_score") or 0.0)

        if rel.get_property("entities_overlap_overlap_score") is not None:
            return float(rel.get_property("entities_overlap_overlap_score") or 0.0)

        return 1.0

    def _leiden_communities(self, knowledge_graph: KnowledgeGraph) -> list[set[Node]]:
        import uuid
        import numpy as np
        from sknetwork.clustering import Leiden
        from sknetwork.data import from_edge_list

        usable_relationships = [
            rel for rel in knowledge_graph.relationships
            if self._relationship_is_usable(rel)
        ]

        if self.debug:
            print("\n[aggregation-leiden] usable relationships:", len(usable_relationships))
            print(
                "[aggregation-leiden] usable relationship distribution:",
                dict(Counter(rel.type for rel in usable_relationships)),
            )

        if not usable_relationships:
            return []

        edge_list = []
        for rel in usable_relationships:
            edge_list.append((rel.source.id.hex, rel.target.id.hex))

            # Ragas relationships often set bidirectional=True.
            # Add reverse edge for community detection when bidirectional.
            if getattr(rel, "bidirectional", False):
                edge_list.append((rel.target.id.hex, rel.source.id.hex))

        graph = from_edge_list(edge_list, directed=True)

        leiden = Leiden(random_state=42)
        labels = leiden.fit_predict(graph["adjacency"])

        community_node_ids = defaultdict(set)

        for label, node_id_hex in zip(labels, graph["names"]):
            community_node_ids[int(label)].add(uuid.UUID(hex=node_id_hex))

        communities = []

        for community_id, node_ids in community_node_ids.items():
            nodes = []
            for node_id in node_ids:
                node = knowledge_graph.get_node_by_id(node_id)
                if node is None:
                    continue
                if node.type.name != "CHUNK":
                    continue
                if not (node.properties.get("page_content") or "").strip():
                    continue
                nodes.append(node)

            if len(nodes) >= self.min_community_size:
                communities.append(set(nodes))

        communities = sorted(communities, key=lambda c: len(c), reverse=True)

        if self.debug:
            sizes = [len(c) for c in communities]
            print("[aggregation-leiden] communities:", len(communities))
            print("[aggregation-leiden] community size distribution:", dict(Counter(sizes)))
            if sizes:
                print("[aggregation-leiden] min size:", min(sizes))
                print("[aggregation-leiden] max size:", max(sizes))
                print("[aggregation-leiden] avg size:", round(sum(sizes) / len(sizes), 2))

        return communities

    def _node_terms(self, nodes: list[Node]) -> tuple[list[str], list[str]]:
        themes = []
        entities = []

        for node in nodes:
            for theme in node.properties.get("themes") or []:
                theme = str(theme).strip()
                if theme:
                    themes.append(theme)

            for entity in node.properties.get("entities") or []:
                entity = str(entity).strip()
                if entity:
                    entities.append(entity)

        def dedupe_keep_order(items):
            out = []
            seen = set()
            for item in items:
                key = item.lower()
                if key in seen:
                    continue
                out.append(item)
                seen.add(key)
                if len(out) >= self.max_terms:
                    break
            return out

        return dedupe_keep_order(themes), dedupe_keep_order(entities)

    def _select_nodes_for_context(self, community_nodes: set[Node]) -> list[Node]:
        nodes = list(community_nodes)

        # Prefer nodes with richer extracted metadata
        nodes = sorted(
            nodes,
            key=lambda node: (
                len(node.properties.get("entities") or []),
                len(node.properties.get("themes") or []),
                len(node.properties.get("page_content") or ""),
            ),
            reverse=True,
        )

        return nodes[: self.max_nodes_per_context]

    def make_contexts(self, scenario: TenderAggregationScenario) -> list[str]:
        contexts = []

        for node in scenario.nodes:
            metadata = node.properties.get("document_metadata", {}) or {}

            chunk_id = metadata.get("chunk_id", "")
            file_name = metadata.get("file_name", "")
            page_number = metadata.get("page_number", "")

            page_content = (node.properties.get("page_content") or "")[
                : self.max_context_chars_per_node
            ]

            context = f"""chunk_id: {chunk_id}
file_name: {file_name}
page_number: {page_number}

{page_content}""".strip()

            contexts.append(context)

        return contexts

    async def _generate_scenarios(
        self,
        n: int,
        knowledge_graph: KnowledgeGraph,
        persona_list: list[Persona],
        callbacks,
    ) -> list[TenderAggregationScenario]:

        if not persona_list:
            raise ValueError(
                "persona_list is empty. Generate personas first or pass a manual persona list."
            )

        communities = self._leiden_communities(knowledge_graph)

        if not communities:
            raise ValueError(
                "No Leiden communities found. Check that KG has summary_similarity or entities_overlap relationships."
            )

        random.shuffle(communities)

        scenarios = []
        skipped = Counter()

        for community_idx, community_nodes in enumerate(communities):
            if len(scenarios) >= n:
                break

            selected_nodes = self._select_nodes_for_context(community_nodes)
            if len(selected_nodes) < self.min_community_size:
                skipped["too_few_selected_nodes"] += 1
                continue

            themes, entities = self._node_terms(selected_nodes)
            terms_for_matching = (themes + entities)[: self.max_terms]

            if not terms_for_matching:
                skipped["no_themes_or_entities"] += 1
                continue

            prompt_input = ThemesPersonasInput(
                themes=terms_for_matching,
                personas=persona_list,
            )

            try:
                persona_concepts = await self.theme_persona_matching_prompt.generate(
                    data=prompt_input,
                    llm=self.llm,
                    callbacks=callbacks,
                )
                persona_mapping = persona_concepts.mapping
            except Exception as e:
                skipped["persona_matching_failed"] += 1
                if self.debug:
                    print("[aggregation-leiden] persona matching failed:", repr(e))
                persona_mapping = {}

            persona_lookup = PersonaList(personas=persona_list)
            valid_personas = []

            for persona_name, concepts in persona_mapping.items():
                concept_l = [str(c).lower() for c in concepts]
                if any(
                    term.lower() in concept or concept in term.lower()
                    for term in terms_for_matching
                    for concept in concept_l
                ):
                    persona = persona_lookup[persona_name]
                    if persona:
                        valid_personas.append(persona)

            if not valid_personas:
                valid_personas = persona_list

            candidates = []

            for task_type in TENDER_AGGREGATION_TASK_TYPES:
                for persona in valid_personas:
                    for style in QueryStyle:
                        for length in QueryLength:
                            candidates.append(
                                TenderAggregationScenario(
                                    community_id=f"leiden:{community_idx}",
                                    nodes=selected_nodes,
                                    task_type=task_type,
                                    themes=themes,
                                    entities=entities,
                                    persona=persona,
                                    style=style,
                                    length=length,
                                )
                            )

            random.shuffle(candidates)

            # Keep only 1-2 scenarios per community to maintain diversity.
            scenarios.extend(candidates[:2])

        scenarios = scenarios[:n]

        if self.debug:
            print("\n[aggregation-leiden] scenario stats")
            print("requested scenarios:", n)
            print("generated scenarios:", len(scenarios))
            print("skipped:", dict(skipped))
            print("task distribution:", dict(Counter(s.task_type for s in scenarios)))
            print("style distribution:", dict(Counter(s.style.name for s in scenarios)))
            print("length distribution:", dict(Counter(s.length.name for s in scenarios)))
            print("community node count distribution:", dict(Counter(len(s.nodes) for s in scenarios)))
            print("sample tasks:", [s.task_type for s in scenarios[:10]])

        return scenarios

    async def _generate_sample(self, scenario, callbacks) -> SingleTurnSample:
        if not isinstance(scenario, TenderAggregationScenario):
            raise TypeError("scenario type should be TenderAggregationScenario")

        reference_contexts = self.make_contexts(scenario)
        context = "\n\n---\n\n".join(reference_contexts)

        prompt_input = TenderAggregationInput(
            persona=scenario.persona,
            task_type=scenario.task_type,
            themes=scenario.themes,
            entities=scenario.entities,
            query_style=scenario.style.value,
            query_length=scenario.length.value,
            context=context,
            llm_context=self.llm_context,
        )

        response = await self.generate_query_reference_prompt.generate(
            data=prompt_input,
            llm=self.llm,
            callbacks=callbacks,
        )

        return SingleTurnSample(
            user_input=response.query,
            reference=response.answer,
            reference_contexts=reference_contexts,
            persona_name=getattr(scenario.persona, "name", None),
            query_style=getattr(scenario.style, "name", None),
            query_length=getattr(scenario.length, "name", None),
        )

In [135]:
leiden_aggregation = LeidenTenderAggregationQuerySynthesizer(
    llm=generator.llm,
    llm_context=generator.llm_context,
)

leiden_aggregation.relationship_types = (
    "summary_similarity",
    "entities_overlap",
)
leiden_aggregation.use_summary_edges = True
leiden_aggregation.use_entity_edges = False
leiden_aggregation.min_summary_similarity = 0.84
leiden_aggregation.min_community_size = 5
leiden_aggregation.max_nodes_per_context = 6
leiden_aggregation.max_context_chars_per_node = 2500
leiden_aggregation.debug = True

query_distribution_aggregation = [
    (leiden_aggregation, 1.0),
]

testset_aggregation = generator.generate(
    testset_size=50,
    query_distribution=query_distribution_aggregation,
    with_debugging_logs=True,
    raise_exceptions=True,
    debug=True,
)

[debug] generate: testset_size = 50
[debug] generate: query_distribution =
         leiden_tender_aggregation_query_synthesizer 1.0
[debug] generate: using existing persona_list
[debug] generate: personas =
         name='EPC Project Tender Manager' role_description='Oversees tender submissions and manages project updates for Engineering, Procurement, and Construction contracts, ensuring compliance with ITT conditions and clarifications.'
         name='Offshore Energy Project Manager' role_description='Oversees the planning, execution, and management of large-scale offshore oil and gas projects, ensuring compliance with contract terms and operational performance standards.'
         name='Procurement Officer' role_description='Manages tender processes, ensures alignment with updates like TB-0012, and oversees communication through platforms such as B2B (ARIBA).'
[debug] generate: scenario splits = [50]
[debug] generate: submit scenario generation leiden_tender_aggregation_query_synthe

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]


[aggregation-leiden] usable relationships: 3071
[aggregation-leiden] usable relationship distribution: {'summary_similarity': 3071}
[aggregation-leiden] communities: 28
[aggregation-leiden] community size distribution: {87: 1, 78: 1, 50: 1, 36: 1, 31: 1, 25: 3, 24: 1, 23: 1, 19: 1, 18: 1, 16: 1, 14: 3, 13: 1, 10: 3, 9: 1, 8: 2, 6: 3, 5: 2}
[aggregation-leiden] min size: 5
[aggregation-leiden] max size: 87
[aggregation-leiden] avg size: 21.25


Generating Scenarios: 100%|██████████| 1/1 [00:58<00:00, 58.50s/it]



[aggregation-leiden] scenario stats
requested scenarios: 50
generated scenarios: 50
skipped: {}
task distribution: {'list_deliverables': 5, 'overview_scope': 3, 'summarize_technical_requirements': 5, 'summarize_contractor_responsibilities': 5, 'summarize_commercial_requirements': 6, 'summarize_approval_requirements': 6, 'summarize_hses_requirements': 3, 'list_requirements': 6, 'summarize_obligations': 3, 'list_documents_or_procedures': 8}
style distribution: {'PERFECT_GRAMMAR': 10, 'MISSPELLED': 16, 'WEB_SEARCH_LIKE': 13, 'POOR_GRAMMAR': 11}
length distribution: {'LONG': 19, 'MEDIUM': 17, 'SHORT': 14}
community node count distribution: {5: 4, 6: 46}
sample tasks: ['list_deliverables', 'overview_scope', 'summarize_technical_requirements', 'list_deliverables', 'summarize_contractor_responsibilities', 'summarize_commercial_requirements', 'summarize_approval_requirements', 'summarize_approval_requirements', 'summarize_technical_requirements', 'overview_scope']
[debug] generate: scenario_s

Generating Samples: 100%|██████████| 50/50 [00:19<00:00,  2.61it/s]

[debug] generate: final samples = 50


In [67]:
from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_aggregation_v1.jsonl")
rows = []
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for sample in testset_aggregation.samples:
        ev = sample.eval_sample

        row = {
            "question": ev.user_input,
            "answer": ev.reference,
            "synthesizer_name": sample.synthesizer_name,
            "persona_name": getattr(ev, "persona_name", None),
            "query_style": getattr(ev, "query_style", None),
            "query_length": getattr(ev, "query_length", None),
            "reference_contexts": ev.reference_contexts,
        }
        rows.append(row)

        f.write(json.dumps(row,indent=2, ensure_ascii=False) + "\n")

print(f"Saved {len(testset_aggregation.samples)} Q/A rows to {OUTPUT_PATH}")

Saved 50 Q/A rows to synthetic_qa_ragas_aggregation_v1.jsonl


In [121]:
## read

from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas_aggregation_v1.jsonl")

text = OUTPUT_PATH.read_text(encoding="utf-8").strip()

rows = []
decoder = json.JSONDecoder()
idx = 0

while idx < len(text):
    # skip whitespace/newlines between objects
    while idx < len(text) and text[idx].isspace():
        idx += 1

    if idx >= len(text):
        break

    obj, end = decoder.raw_decode(text, idx)
    rows.append(obj)
    idx = end

print("loaded rows:", len(rows))
print(rows[0].keys())

loaded rows: 50
dict_keys(['question', 'answer', 'synthesizer_name', 'persona_name', 'query_style', 'query_length', 'reference_contexts'])


In [122]:
import json
import re
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage

In [136]:
SYNTHETIC_QA_JUDGE_PROMPT = """
You are a strict evaluator for synthetic QA data used to evaluate a tender/EPC RAG system.

You must judge whether the generated question-answer pair is useful, faithful, and relevant.

Evaluate using ONLY the provided reference contexts.

Scoring scale for each metric:
- 5 = excellent
- 4 = good
- 3 = acceptable but imperfect
- 2 = weak
- 1 = bad
- 0 = unusable

Metrics:

1. faithfulness
Does the answer only contain information supported by the reference contexts?
Penalize invented facts, unsupported explanations, wrong parties, wrong dates, wrong clauses, or over-generalization.

2. question_relevance
Is the question relevant to the reference contexts?
Penalize questions that are too generic, unrelated, impossible to answer from the context, or not tender/EPC relevant.

3. answer_relevance
Does the answer directly answer the question?
Penalize evasive, incomplete, overly broad, or off-topic answers.

4. context_sufficiency
Do the reference contexts contain enough evidence to answer the question?
Penalize if the answer requires missing context.

5. aggregation_quality
If the query is list/summary/overview/aggregation style, does it actually require multiple facts?
If it is not aggregation-style, score based on whether the question type is clear and useful.

Return STRICT JSON only:
{
  "faithfulness": 0,
  "question_relevance": 0,
  "answer_relevance": 0,
  "context_sufficiency": 0,
  "aggregation_quality": 0,
  "overall_score": 0,
  "verdict": "keep|review|drop",
  "issues": ["..."],
  "explanation": "short explanation"
}
""".strip()

In [137]:
def extract_json_object(text: str) -> dict:
    text = (text or "").strip()

    # Remove markdown fences if present
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    # Fallback: extract first JSON object
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response: {text[:500]}")

    return json.loads(match.group(0))


def sample_to_dict(sample_obj):
    """
    Supports:
    - Ragas TestsetSample
    - SingleTurnSample
    - already flattened dict
    """
    if isinstance(sample_obj, dict):
        return sample_obj

    # Ragas TestsetSample has .eval_sample
    eval_sample = getattr(sample_obj, "eval_sample", sample_obj)

    return {
        "question": getattr(eval_sample, "user_input", None),
        "answer": getattr(eval_sample, "reference", None),
        "reference_contexts": getattr(eval_sample, "reference_contexts", None) or [],
        "synthesizer_name": getattr(sample_obj, "synthesizer_name", None),
        "persona_name": getattr(eval_sample, "persona_name", None),
        "query_style": getattr(eval_sample, "query_style", None),
        "query_length": getattr(eval_sample, "query_length", None),
    }

In [138]:
async def judge_synthetic_qa(row: dict, llm, max_context_chars: int = 12000) -> dict:
    question = row.get("question") or row.get("user_input") or ""
    answer = row.get("answer") or row.get("reference") or ""
    reference_contexts = row.get("reference_contexts") or []

    context = "\n\n---\n\n".join(str(c) for c in reference_contexts)
    context = context[:max_context_chars]

    user_prompt = f"""
Question:
{question}

Answer:
{answer}

Reference contexts:
{context}
""".strip()

    resp = await llm.ainvoke([
        SystemMessage(content=SYNTHETIC_QA_JUDGE_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    data = extract_json_object(resp.content)

    scores = [
        float(data.get("faithfulness", 0) or 0),
        float(data.get("question_relevance", 0) or 0),
        float(data.get("answer_relevance", 0) or 0),
        float(data.get("context_sufficiency", 0) or 0),
        float(data.get("aggregation_quality", 0) or 0),
    ]

    if not data.get("overall_score"):
        data["overall_score"] = round(sum(scores) / len(scores), 2)

    if not data.get("verdict"):
        if data["overall_score"] >= 4.0 and min(scores) >= 3:
            data["verdict"] = "keep"
        elif data["overall_score"] >= 3.0:
            data["verdict"] = "review"
        else:
            data["verdict"] = "drop"

    return data

In [139]:

samples = testset_aggregation.samples

qa_rows = [sample_to_dict(s) for s in samples]

print("samples to judge:", len(qa_rows))

samples to judge: 50


In [140]:
judge_results = []

for i, row in enumerate(qa_rows, start=1):
    print(f"[judge] {i}/{len(qa_rows)}")

    try:
        judgment = await judge_synthetic_qa(row, llm=llm)

        judge_results.append({
            **row,
            **judgment,
        })

    except Exception as e:
        print("[judge-error]", repr(e))
        judge_results.append({
            **row,
            "faithfulness": 0,
            "question_relevance": 0,
            "answer_relevance": 0,
            "context_sufficiency": 0,
            "aggregation_quality": 0,
            "overall_score": 0,
            "verdict": "drop",
            "issues": [f"judge_failed: {repr(e)}"],
            "explanation": "Judge failed.",
        })

judge_df = pd.DataFrame(judge_results)
judge_df.head()

[judge] 1/50
[judge] 2/50
[judge] 3/50
[judge] 4/50
[judge] 5/50
[judge] 6/50
[judge] 7/50
[judge] 8/50
[judge] 9/50
[judge] 10/50
[judge] 11/50
[judge] 12/50
[judge] 13/50
[judge] 14/50
[judge] 15/50
[judge] 16/50
[judge] 17/50
[judge] 18/50
[judge] 19/50
[judge] 20/50
[judge] 21/50
[judge] 22/50
[judge] 23/50
[judge] 24/50
[judge] 25/50
[judge] 26/50
[judge] 27/50
[judge] 28/50
[judge] 29/50
[judge] 30/50
[judge] 31/50
[judge] 32/50
[judge] 33/50
[judge] 34/50
[judge] 35/50
[judge] 36/50
[judge] 37/50
[judge] 38/50
[judge] 39/50
[judge] 40/50
[judge] 41/50
[judge] 42/50
[judge] 43/50
[judge] 44/50
[judge] 45/50
[judge] 46/50
[judge] 47/50
[judge] 48/50
[judge] 49/50
[judge] 50/50


,question,answer,reference_contexts,synthesizer_name,persona_name,query_style,query_length,faithfulness,question_relevance,answer_relevance,context_sufficiency,aggregation_quality,overall_score,verdict,issues,explanation
0,Can you provide a comprehensive list of all el...,The Contractor is required to supply the follo...,[chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_...,leiden_tender_aggregation_query_synthesizer,EPC Project Tender Manager,PERFECT_GRAMMAR,LONG,3,4,3,3,4,17,review,[Some deliverables listed in the answer are no...,The question is relevant and the answer attemp...
1,Can you provide an overview of the scope of el...,The scope of electrical documentation and deli...,[chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_...,leiden_tender_aggregation_query_synthesizer,Offshore Energy Project Manager,MISSPELLED,MEDIUM,3,4,3,3,4,17,review,[Answer includes items not explicitly listed i...,The question is relevant to the provided conte...
2,What are the technical requirements for the AG...,The technical requirements for the specified e...,[chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_...,leiden_tender_aggregation_query_synthesizer,Procurement Officer,WEB_SEARCH_LIKE,LONG,4,5,4,4,5,22,review,[The answer includes identification codes and ...,The question is highly relevant and the answer...
3,Can you list all the deliverables related to t...,The deliverables related to the specified equi...,[chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_...,leiden_tender_aggregation_query_synthesizer,Offshore Energy Project Manager,MISSPELLED,MEDIUM,2,4,3,3,4,16,review,[Some identification codes in the answer are n...,The question is relevant and aggregation-style...
4,What are the Contractor's responsibilities for...,The Contractor's responsibilities include:\n\n...,[chunk_id: ROC_INPEX_ITT_Ichthys_Onshore_AGRU_...,leiden_tender_aggregation_query_synthesizer,Procurement Officer,POOR_GRAMMAR,LONG,5,5,5,5,5,25,keep,[],The answer is fully supported by the reference...


In [ ]:
judge_df

In [141]:
# ---------------------------------------------------------------------
# Filter kept aggregation rows
# ---------------------------------------------------------------------

kept_aggregation_rows = [
    row for row in judge_results
    if row.get("verdict") == "keep"
    and float(row.get("faithfulness", 0) or 0) >= 4
    and float(row.get("question_relevance", 0) or 0) >= 4
    and float(row.get("answer_relevance", 0) or 0) >= 4
    and float(row.get("context_sufficiency", 0) or 0) >= 4
    and float(row.get("aggregation_quality", 0) or 0) >= 4
]

print("judged aggregation rows:", len(judge_results))
print("kept aggregation rows:", len(kept_aggregation_rows))

judged aggregation rows: 50
kept aggregation rows: 25


In [142]:
# ---------------------------------------------------------------------
# Save judged + kept aggregation rows
# ---------------------------------------------------------------------

from pathlib import Path
import json

OUTPUT_JUDGED_PATH = Path("synthetic_qa_ragas_aggregation_judged_v2.jsonl")
OUTPUT_KEEP_PATH = Path("synthetic_qa_ragas_aggregation_kept_v2.jsonl")

with OUTPUT_JUDGED_PATH.open("w", encoding="utf-8") as f:
    for row in judge_results:
        f.write(json.dumps(row, ensure_ascii=False,indent = 2) + "\n")

with OUTPUT_KEEP_PATH.open("w", encoding="utf-8") as f:
    for row in kept_aggregation_rows:
        f.write(json.dumps(row, ensure_ascii=False,indent =2) + "\n")

print(f"saved {len(judge_results)} judged rows to {OUTPUT_JUDGED_PATH}")
print(f"saved {len(kept_aggregation_rows)} kept rows to {OUTPUT_KEEP_PATH}")

saved 50 judged rows to synthetic_qa_ragas_aggregation_judged_v2.jsonl
saved 25 kept rows to synthetic_qa_ragas_aggregation_kept_v2.jsonl


### Save to Neo4J

In [ ]:
from datetime import datetime
import json
import math
from neo4j import GraphDatabase

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------

RAGAS_KG_RUN_ID = datetime.now().strftime("ragas_synth_%Y%m%d_%H%M%S")

SAVE_EMBEDDINGS = True  # set False if you do not want summary_embedding in Neo4j

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "neo4j123"

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    connection_timeout=10,
    max_connection_lifetime=600,
)

kg = generator.knowledge_graph

print("Ragas KG run_id:", RAGAS_KG_RUN_ID)
print("nodes:", len(kg.nodes))
print("relationships:", len(kg.relationships))

Ragas KG run_id: ragas_synth_20260526_132243
nodes: 1000
relationships: 30633


In [ ]:
# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def _json_safe(value):
    """Make nested values JSON serializable."""
    if value is None:
        return None

    if isinstance(value, (str, int, bool)):
        return value

    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value

    if isinstance(value, (list, tuple, set)):
        return [_json_safe(v) for v in value]

    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}

    return str(value)


def _is_neo4j_primitive(value):
    return value is None or isinstance(value, (str, int, float, bool))


def _is_neo4j_primitive_list(value):
    return (
        isinstance(value, list)
        and all(_is_neo4j_primitive(v) for v in value)
    )


def sanitize_node_properties(properties: dict, save_embeddings: bool = True) -> dict:
    """
    Flatten useful Ragas node metadata into Neo4j-friendly properties.
    Keep nested metadata JSON only as backup/debug.
    """
    properties = properties or {}
    out = {}

    # Main text
    page_content = properties.get("page_content")
    if page_content is not None:
        out["pageContent"] = str(page_content)

    # Summary / themes / entities
    if properties.get("summary") is not None:
        out["summary"] = str(properties.get("summary"))

    if properties.get("themes") is not None:
        out["themes"] = [str(x) for x in properties.get("themes", [])]

    if properties.get("entities") is not None:
        out["entities"] = [str(x) for x in properties.get("entities", [])]

    # Optional embedding
    if save_embeddings and properties.get("summary_embedding") is not None:
        out["summaryEmbedding"] = [
            float(x) for x in properties.get("summary_embedding", [])
        ]

    # Flatten document metadata
    metadata = properties.get("document_metadata") or {}

    out["chunkId"] = metadata.get("chunk_id")
    out["fileName"] = metadata.get("file_name")
    out["pageNumber"] = metadata.get("page_number")
    out["docURL"] = metadata.get("doc_url")
    out["sequenceNumber"] = metadata.get("sequence_number")
    out["sectionHint"] = metadata.get("section_hint")

    # Metadata may include nested entity dicts etc.; keep backup JSON
    out["documentMetadataJson"] = json.dumps(
        _json_safe(metadata),
        ensure_ascii=False,
    )

    # Optional full backup. You can remove this if too large.
    out["propertiesJson"] = json.dumps(
        _json_safe({
            k: v
            for k, v in properties.items()
            if save_embeddings or k != "summary_embedding"
        }),
        ensure_ascii=False,
    )

    # Remove None values because Neo4j does not store them usefully
    out = {k: v for k, v in out.items() if v is not None}

    return out

def sanitize_relationship_properties(properties: dict) -> dict:
    out = {}

    for key, value in (properties or {}).items():
        key = str(key)
        safe_value = _json_safe(value)

        if _is_neo4j_primitive(safe_value) or _is_neo4j_primitive_list(safe_value):
            out[key] = safe_value
        else:
            out[f"{key}_json"] = json.dumps(safe_value, ensure_ascii=False)

    out["propertiesJson"] = json.dumps(_json_safe(properties or {}), ensure_ascii=False)
    return out

In [ ]:
# ---------------------------------------------------------------------
# CREATE CONSTRAINTS / INDEXES
# ---------------------------------------------------------------------

with neo4j_driver.session() as session:
    session.run("""
        CREATE CONSTRAINT ragas_node_id_unique IF NOT EXISTS
        FOR (n:RagasNode) REQUIRE n.ragasNodeId IS UNIQUE
    """)

    session.run("""
        CREATE INDEX ragas_node_run_id IF NOT EXISTS
        FOR (n:RagasNode) ON (n.runId)
    """)

    session.run("""
        CREATE INDEX ragas_node_type IF NOT EXISTS
        FOR (n:RagasNode) ON (n.nodeType)
    """)

    session.run("""
        CREATE INDEX ragas_relation_id IF NOT EXISTS
        FOR ()-[r:RAGAS_RELATION]-() ON (r.ragasRelationshipId)
    """)

print("Neo4j constraints/indexes ensured")

Neo4j constraints/indexes ensured


In [ ]:
def parse_themes_response(raw: str, max_themes: int = 10) -> list[str]:
    raw = (raw or "").strip()

    # Remove markdown code fences
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw)

    # Try direct JSON
    try:
        data = json.loads(raw)
        themes = data.get("themes", [])
    except Exception:
        # Try to extract JSON object from surrounding text
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if match:
            try:
                data = json.loads(match.group(0))
                themes = data.get("themes", [])
            except Exception:
                themes = []
        else:
            themes = []

    # Last fallback: line cleanup
    if not themes:
        themes = [
            line.strip("-•, ").strip().strip('"').strip("'")
            for line in raw.splitlines()
            if line.strip()
            and not line.strip().startswith("{")
            and not line.strip().startswith("}")
            and "themes" not in line.lower()
        ]

    clean = []
    seen = set()

    for theme in themes:
        theme = str(theme).strip().strip(",").strip().strip('"').strip("'")
        if not theme:
            continue

        # Remove JSON syntax leftovers
        if theme in {"{", "}", "[", "]", "```json", "```"}:
            continue
        if theme.lower().startswith("themes"):
            continue

        key = theme.lower()
        if key in seen:
            continue

        clean.append(theme)
        seen.add(key)

        if len(clean) >= max_themes:
            break

    return clean

In [ ]:
for node in kg.nodes:
    themes = node.properties.get("themes")
    if isinstance(themes, list):
        raw = "\n".join(str(x) for x in themes)
        node.properties["themes"] = parse_themes_response(raw, max_themes=10)

In [ ]:
node_rows = []

for node in kg.nodes:
    props = sanitize_node_properties(
        node.properties,
        save_embeddings=SAVE_EMBEDDINGS,
    )

    node_rows.append({
        "ragas_node_id": str(node.id),
        "run_id": RAGAS_KG_RUN_ID,
        "node_type": node.type.name,
        "props": props,
    })
    themes = node.properties.get("themes")
    if isinstance(themes, list):
        raw = "\n".join(str(x) for x in themes)
        node.properties["themes"] = parse_themes_response(raw, max_themes=10)

with neo4j_driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MERGE (n:RagasNode {ragasNodeId: row.ragas_node_id})
        SET n:RagasKG,
            n.runId = row.run_id,
            n.nodeType = row.node_type
        SET n += row.props

        WITH n, row
        FOREACH (_ IN CASE WHEN row.node_type = 'CHUNK' THEN [1] ELSE [] END |
            SET n:RagasChunk
        )
        FOREACH (_ IN CASE WHEN row.node_type = 'DOCUMENT' THEN [1] ELSE [] END |
            SET n:RagasDocument
        )
    """, {"rows": node_rows})

print(f"Saved {len(node_rows)} Ragas nodes")

Saved 1000 Ragas nodes


In [ ]:
semantic_rows = []
entity_overlap_rows = []

for rel in kg.relationships:
    source_id = str(rel.source.id)
    target_id = str(rel.target.id)

    if rel.type == "summary_similarity":
        semantic_rows.append({
            "source_id": source_id,
            "target_id": target_id,
            "run_id": RAGAS_KG_RUN_ID,
            "score": float(rel.properties.get("summary_similarity", 0.0)),
        })

    elif rel.type == "entities_overlap":
        entity_overlap_rows.append({
            "source_id": source_id,
            "target_id": target_id,
            "run_id": RAGAS_KG_RUN_ID,
            "score": float(rel.properties.get("entities_overlap_score", 0.0)),
            "overlapped_items_json": json.dumps(
                rel.properties.get("overlapped_items", []),
                ensure_ascii=False,
            ),
        })

In [ ]:
with neo4j_driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (source:RagasNode {ragasNodeId: row.source_id})
        MATCH (target:RagasNode {ragasNodeId: row.target_id})
        MERGE (source)-[r:SEMANTIC_SIMILAR]->(target)
        SET r.runId = row.run_id,
            r.score = row.score,
            r.source = "ragas_summary_embedding"
    """, {"rows": semantic_rows})

In [ ]:
with neo4j_driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (source:RagasNode {ragasNodeId: row.source_id})
        MATCH (target:RagasNode {ragasNodeId: row.target_id})
        MERGE (source)-[r:ENTITY_OVERLAP]->(target)
        SET r.runId = row.run_id,
            r.score = row.score,
            r.overlappedItemsJson = row.overlapped_items_json,
            r.source = "ragas_entity_overlap"
    """, {"rows": entity_overlap_rows})

In [ ]:
from collections import defaultdict
import json

combined = {}

for rel in kg.relationships:
    source_id = str(rel.source.id)
    target_id = str(rel.target.id)

    # normalize direction for bidirectional chunk similarity
    pair = tuple(sorted([source_id, target_id]))

    if pair not in combined:
        combined[pair] = {
            "pair_id": f"{pair[0]}::{pair[1]}",
            "run_id": RAGAS_KG_RUN_ID,
            "source_id": pair[0],
            "target_id": pair[1],
            "summary_similarity": None,
            "entities_overlap_score": None,
            "overlapped_items_json": None,
            "relationship_types": [],
        }

    row = combined[pair]
    row["relationship_types"].append(rel.type)

    if rel.type == "summary_similarity":
        row["summary_similarity"] = float(
            rel.properties.get("summary_similarity", 0.0)
        )

    elif rel.type == "entities_overlap":
        row["entities_overlap_score"] = float(
            rel.properties.get("entities_overlap_score", 0.0)
        )
        row["overlapped_items_json"] = json.dumps(
            rel.properties.get("overlapped_items", []),
            ensure_ascii=False,
        )

combined_relationship_rows = list(combined.values())

for row in combined_relationship_rows:
    row["relationship_types"] = sorted(set(row["relationship_types"]))

    # Optional combined score
    summary = row["summary_similarity"] or 0.0
    overlap = row["entities_overlap_score"] or 0.0

    row["combined_score"] = (
        0.7 * summary +
        0.3 * overlap
    )

In [ ]:
combined_relationship_rows

[{'pair_id': '06d2f24c-1245-4014-9b5c-27db50a4ba8f::4590863a-e0ae-44bf-91a0-2d844b16d7ba',
  'run_id': 'ragas_synth_20260526_132243',
  'source_id': '06d2f24c-1245-4014-9b5c-27db50a4ba8f',
  'target_id': '4590863a-e0ae-44bf-91a0-2d844b16d7ba',
  'summary_similarity': 0.7441028169534835,
  'entities_overlap_score': 0.075,
  'overlapped_items_json': '[["Value Enhancement (VE) process", "Value Enhancement Opportunities Briefing"], ["Section 3 of the ITT Conditions of Tender", "Section 3 of the ITT Conditions of Tender"], ["Tender Bulletin 004", "Tender Bulletin 004"]]',
  'relationship_types': ['entities_overlap', 'summary_similarity'],
  'combined_score': 0.5433719718674384},
 {'pair_id': '06d2f24c-1245-4014-9b5c-27db50a4ba8f::29aac973-1ea2-4ce7-b4a8-56e4f2e29b78',
  'run_id': 'ragas_synth_20260526_132243',
  'source_id': '06d2f24c-1245-4014-9b5c-27db50a4ba8f',
  'target_id': '29aac973-1ea2-4ce7-b4a8-56e4f2e29b78',
  'summary_similarity': 0.7007093078797674,
  'entities_overlap_score': N

In [ ]:
with neo4j_driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (source:RagasNode {ragasNodeId: row.source_id})
        MATCH (target:RagasNode {ragasNodeId: row.target_id})

        MERGE (source)-[r:RAGAS_CHUNK_SIMILARITY {pairId: row.pair_id}]->(target)
        SET r.runId = row.run_id,
            r.relationshipTypes = row.relationship_types,
            r.summarySimilarity = row.summary_similarity,
            r.entitiesOverlapScore = row.entities_overlap_score,
            r.overlappedItemsJson = row.overlapped_items_json,
            r.combinedScore = row.combined_score,
            r.hasSummarySimilarity = row.summary_similarity IS NOT NULL,
            r.hasEntityOverlap = row.entities_overlap_score IS NOT NULL
    """, {"rows": combined_relationship_rows})

print(f"Saved {len(combined_relationship_rows)} combined chunk similarity relationships")

Saved 28015 combined chunk similarity relationships


In [ ]:
# ---------------------------------------------------------------------
# WRITE TESTSET SAMPLES TOO
# ---------------------------------------------------------------------

sample_rows = []

for idx, sample in enumerate(testset.samples):
    ev = sample.eval_sample

    sample_rows.append({
        "sample_id": f"{RAGAS_KG_RUN_ID}::sample::{idx}",
        "run_id": RAGAS_KG_RUN_ID,
        "idx": idx,
        "question": ev.user_input,
        "reference_answer": ev.reference,
        "synthesizer_name": sample.synthesizer_name,
        "persona_name": getattr(ev, "persona_name", None),
        "query_style": getattr(ev, "query_style", None),
        "query_length": getattr(ev, "query_length", None),
        "reference_contexts": ev.reference_contexts or [],
    })

with neo4j_driver.session() as session:
    session.run("""
        CREATE CONSTRAINT ragas_sample_id_unique IF NOT EXISTS
        FOR (s:RagasSample) REQUIRE s.sampleId IS UNIQUE
    """)

    session.run("""
        UNWIND $rows AS row
        MERGE (s:RagasSample {sampleId: row.sample_id})
        SET s.runId = row.run_id,
            s.idx = row.idx,
            s.question = row.question,
            s.referenceAnswer = row.reference_answer,
            s.synthesizerName = row.synthesizer_name,
            s.personaName = row.persona_name,
            s.queryStyle = row.query_style,
            s.queryLength = row.query_length,
            s.referenceContexts = row.reference_contexts
    """, {"rows": sample_rows})

print(f"Saved {len(sample_rows)} Ragas samples")

In [ ]:
# ---------------------------------------------------------------------
# QUICK CHECK
# ---------------------------------------------------------------------

with neo4j_driver.session() as session:
    summary = session.run("""
        MATCH (n:RagasNode {runId: $run_id})
        WITH count(n) AS nodes
        MATCH ()-[r:RAGAS_RELATION {runId: $run_id}]->()
        WITH nodes, count(r) AS relationships
        MATCH (s:RagasSample {runId: $run_id})
        RETURN nodes, relationships, count(s) AS samples
    """, {"run_id": RAGAS_KG_RUN_ID}).single()

print(dict(summary))

In [ ]:
testset.samples[1].eval_sample

SingleTurnSample(user_input='What VE process do?', retrieved_contexts=None, reference_contexts=['Tender No.  WS33071728 - Ichthys Onshore AGRU Upgrade and CCS Project - \nEngineering, Procurement and Construction Contract (EPC ITT) \n \nReference: \nWS33071728 / TB / 0004 \nDate:  \n02 May 2025 \n \nSubject: \nCompany FEED value enhancement optimisation update \n \nTender Bulletin No. 0004 \n \nDear Tenderer, \n \nAs Tenderer has been made aware, Company has commenced an internal value \nenhancement exercise (VE) for the Project.   \n \nThe purpose of the VE is for Company to identify, assess and select potential value \nenhancement improvement opportunities (including but not limited to cost and schedule \nconsiderations) for further development following a structured process. \n \nCompany’s VE process has been developed using the following categorisation waterfall for \npotential enhancement options; \n \nCategory Brief Description \n1 \nPotential opportunities to be developed by Ten

In [ ]:
from pathlib import Path
import json

OUTPUT_PATH = Path("synthetic_qa_ragas.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for sample in testset.samples:
        ev = sample.eval_sample

        row = {
            "question": ev.user_input,
            "answer": ev.reference,
            "synthesizer_name": sample.synthesizer_name,
            "persona_name": getattr(ev, "persona_name", None),
            "query_style": getattr(ev, "query_style", None),
            "query_length": getattr(ev, "query_length", None),
            "reference_contexts": ev.reference_contexts,
        }

        f.write(json.dumps(row,indent=2, ensure_ascii=False) + "\n")

print(f"Saved {len(testset.samples)} Q/A rows to {OUTPUT_PATH}")

Saved 4 Q/A rows to synthetic_qa_ragas.jsonl
